# MSU FRUIT · Fine-tune SAM 2.1 on our vineyard video, and score any model on the same clips

This notebook is one pipeline with two jobs:

1. **Train**: fine-tune Meta's SAM 2.1 (the newest SAM 2 release) on grape clusters that we label in our own videos.
2. **Evaluate**: grade *any* model on the same held-out clips with the same rules. SAM 2.1 before and after fine-tuning, YOLO / YOLOE, SAM 3, or anything else that can write a mask.

```
videos ──▶ short clips ──▶ labels ──▶ score the starting models ──▶ fine-tune SAM 2.1 ──▶ score again ──▶ run on new video
 Step 1      Step 2        Step 3            Step 4                      Step 5             Step 6          Step 7
```

## How to use it

**First run: the demo.** Make sure the runtime has a GPU (**Runtime ▸ Change runtime type ▸ T4 GPU**), leave `DEMO` ticked in cell A and choose **Runtime ▸ Run all**. The notebook downloads eight public grape photos with hand-drawn cluster masks, turns them into short panning clips, scores the pretrained models, fine-tunes for a few steps and scores again. Nothing of ours is touched. If the demo reaches Step 7, the setup works.

**Real run: our videos.**
1. In cell A untick `DEMO`, keep `SAVE_TO_DRIVE` ticked, then **Runtime ▸ Run all**. Approve the Google Drive prompt.
2. **Step 1** shows the `videos` folder. Put the videos there (upload button, drag and drop, or straight into Drive), then re-run Step 1 and Step 2.
3. **Step 3** is the only manual work: draw one box per grape cluster in each clip, check the result, press *mark done*.
4. From **Step 3c** on, choose **Runtime ▸ Run after** to score, train, score again and produce the overlay video.

Everything lands in one project folder: `videos/`, `dataset/` (frames and labels), `predictions/` (what each model drew), `models/` (fine-tuned weights), `results/` (leaderboards and videos).

**On the MSU HPCC** the same notebook runs in OnDemand Jupyter (set `PROJECT_DIR` in cell A), and long jobs go to the queue with `sbatch hpcc/run.sb train …` / `evaluate` / `video …`. Both use the same project folder layout and the same code. See Appendix E and `sam2_pipeline/README.md`.

## Three things to know before trusting a number

- **SAM 2 does not find grapes. It follows what it is shown.** Give it a box or a click on a cluster and it draws the mask and tracks it. So the leaderboard has two tables: *prompted* models (shown each cluster once) and *automatic* models (given only pixels). The automatic SAM 2 pipeline here is "a detector finds the clusters, SAM 2.1 segments and tracks them".
- **Scores are only as honest as the val clips.** Val clips must come from video the model never trained on, and every cluster in them must be labelled. Labels drafted by a model favour that model, which is why drafts stay unused until a person approves them.
- **Fine-tuning is not guaranteed to help.** It can only learn from the places where you corrected pretrained SAM 2.1 while labelling. Step 6 puts pretrained and fine-tuned side by side on the same clips so the answer is measured, not assumed.

*SAM 2.1 is the newest model in the SAM 2 line (checkpoints Sept 2024, code last updated Dec 2024). SAM 3 / SAM 3.1 (Nov 2025 / Mar 2026) is a newer, much larger family that also takes text prompts such as "grape cluster". This notebook trains SAM 2.1 and can score SAM 3.1 next to it (Appendix B).*

## Setup
Run cells A and B once per session; cell A is the only one with choices to make. Then run the **Tools** cells below them (they only define functions, so there is nothing to read unless you are curious). **Runtime ▸ Run all** does all of this.

In [ ]:
#@title A · Settings  { display-mode: "form" }
#@markdown **Run this first.** Leave `DEMO` on for your first run: it uses public sample grape
#@markdown clips so you can watch every step work before you add your own videos.

PROJECT_NAME  = "msu_fruit_sam2"  #@param {type:"string"}
DEMO          = True              #@param {type:"boolean"}
SAVE_TO_DRIVE = True              #@param {type:"boolean"}
SAM2_SIZE     = "base_plus"       #@param ["tiny", "small", "base_plus", "large"]
OBJECT_NAME   = "grape cluster"   #@param {type:"string"}
PROJECT_DIR   = ""                #@param {type:"string"}

#@markdown ---
#@markdown `SAVE_TO_DRIVE` keeps videos, labels and trained weights in *My Drive / PROJECT_NAME* so a
#@markdown Colab disconnect does not erase your labelling. The demo never touches Drive.
#@markdown
#@markdown `PROJECT_DIR`: leave empty in Colab. On the MSU HPCC (OnDemand Jupyter) put the same folder
#@markdown as `PROJECT_DIR` in `hpcc/config.sh`, e.g. `~/msu_fruit_project`, so the notebook and the
#@markdown batch jobs share one project.

In [ ]:
#@title B · Install and set up (about a minute)  { display-mode: "form" }
import os, sys, subprocess, json, time, math, random, shutil, re, glob, io, base64, warnings, functools
from pathlib import Path
from dataclasses import dataclass, field

IN_COLAB = "google.colab" in sys.modules
# Work folder for the SAM 2 code, pretrained weights and temporary files (never your labels):
# $FRUIT_BASE if set, /content in Colab, otherwise ~/.fruit_sam2 (what hpcc/setup_env.sh fills in).
BASE = Path(os.environ.get("FRUIT_BASE") or ("/content" if IN_COLAB else Path.home() / ".fruit_sam2")).expanduser()
BASE.mkdir(parents=True, exist_ok=True)

def sh(cmd, check=True, quiet=True):
    """Run a command (a list of arguments, so paths with spaces are safe); show its output only when it fails."""
    r = subprocess.run([str(c) for c in cmd], text=True, capture_output=True)
    if r.returncode != 0 and check:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"command failed: {' '.join(map(str, cmd))}")
    if not quiet:
        print(r.stdout)
    return r

def _missing(*mods):
    import importlib.util
    return [m for m in mods if importlib.util.find_spec(m) is None]

# --- SAM 2 code, pinned to the exact commit this pipeline was tested with -------------------
SAM2_COMMIT = "2b90b9f5ceec907a1c18123530e92e794ad901a4"   # facebookresearch/sam2, 2024-12-15 (latest)
SAM2_REPO = Path(os.environ.get("FRUIT_SAM2_REPO") or BASE / "sam2_repo")
if not (SAM2_REPO / "sam2" / "build_sam.py").exists():
    shutil.rmtree(SAM2_REPO, ignore_errors=True)          # a half-finished earlier download starts over
    SAM2_REPO.mkdir(parents=True, exist_ok=True)
    sh(["git", "-C", SAM2_REPO, "init", "-q"])
    sh(["git", "-C", SAM2_REPO, "remote", "add", "origin", "https://github.com/facebookresearch/sam2.git"])
    sh(["git", "-C", SAM2_REPO, "fetch", "-q", "--depth", "1", "origin", SAM2_COMMIT])
    sh(["git", "-C", SAM2_REPO, "checkout", "-q", "FETCH_HEAD"])
if str(SAM2_REPO) not in sys.path:
    sys.path.insert(0, str(SAM2_REPO))        # makes `sam2` and `training` importable, no pip build

# --- Python packages (only what is missing; Colab already has most) --------------------------
_pip = {"hydra": "hydra-core>=1.3.2", "iopath": "iopath>=0.1.10", "fvcore": "fvcore",
        "tensordict": "tensordict>=0.6.0", "submitit": "submitit", "pycocotools": "pycocotools",
        "tensorboard": "tensorboard", "scipy": "scipy", "yaml": "pyyaml",
        "pandas": "pandas", "matplotlib": "matplotlib", "cv2": "opencv-python-headless",
        "anywidget": "anywidget", "ipywidgets": "ipywidgets", "tqdm": "tqdm"}
_need = [_pip[m] for m in _missing(*_pip)]
if _need:
    print("installing:", ", ".join(_need))
    sh([sys.executable, "-m", "pip", "install", "-q", *_need])
    import importlib; importlib.invalidate_caches()

import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display, HTML, Video, clear_output

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    _p = torch.cuda.get_device_properties(0)
    GPU_GB, GPU_NAME = _p.total_memory / 2**30, _p.name
    BF16_OK = _p.major >= 8          # A100 / L4 / L40S / H100 / H200. T4 and V100 train in float16 instead.
    if _p.major >= 8:                # same speed-up the official notebooks use
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
else:
    GPU_GB, GPU_NAME, BF16_OK = 0.0, "", False

def _in_notebook():
    try:
        return get_ipython().__class__.__name__ in ("ZMQInteractiveShell", "Shell")   # Jupyter / Colab
    except NameError:
        return False

# --- Project folder ------------------------------------------------------------------------------
def _safe(name):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(name)).strip("_")

def use_project(root):
    """Point every tool at one project folder: videos/, dataset/ (frames + labels), predictions/,
    models/, results/, runs/. The notebook and the HPCC command line both go through here."""
    global ROOT, VIDEOS, DATA, IMG, ANN, PRED, RUNS, WEIGHTS, RESULTS, _META, PROMPTS
    ROOT = Path(root).expanduser().resolve()
    VIDEOS, DATA      = ROOT / "videos", ROOT / "dataset"
    IMG, ANN          = DATA / "JPEGImages", DATA / "Annotations"
    PRED, RUNS        = ROOT / "predictions", ROOT / "runs"
    WEIGHTS, RESULTS  = ROOT / "models", ROOT / "results"
    _META, PROMPTS    = DATA / "clips.json", DATA / "prompts"
    for _d in (VIDEOS, IMG, ANN, PRED, RUNS, WEIGHTS, RESULTS):
        _d.mkdir(parents=True, exist_ok=True)
    return ROOT

CKPT_DIR = Path(os.environ.get("FRUIT_CHECKPOINTS") or BASE / "checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if not globals().get("_DEFER_PROJECT"):           # the command-line module picks its folder later
    _project = _safe(PROJECT_NAME) or "fruit_sam2"
    _parent = Path(os.environ.get("FRUIT_PROJECTS") or ("/content" if IN_COLAB else Path.home()))
    if DEMO:        # the demo always gets its own folder, never your real project
        _root = (Path(PROJECT_DIR).expanduser().parent if PROJECT_DIR else _parent) / f"{_project}_demo"
    elif PROJECT_DIR:
        _root = PROJECT_DIR
    elif IN_COLAB and SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        _root = Path("/content/drive/MyDrive") / _project
    else:
        _root = _parent / _project
    use_project(_root)
    print(f"torch {torch.__version__} · " + (f"GPU {GPU_NAME} ({GPU_GB:.0f} GB)" if DEVICE == "cuda" else "CPU")
          + f" · SAM 2 code @ {SAM2_COMMIT[:7]}")
    if DEVICE != "cuda":
        print("⚠️  No GPU. Everything still runs, but slowly. In Colab: Runtime ▸ Change runtime type ▸ T4 GPU.")
    print(("🧪 DEMO MODE: public sample grapes, not your videos.  " if DEMO else "") + f"Project folder: {ROOT}")

## Tools
The cells below define every function the steps use: clips and label files, SAM 2.1, labelling, imports, scoring, model wrappers, training, comparisons. The HPCC command line (`fruit.py`) runs exactly this code: `source/build.py` copies these cells into `fruit_sam2.py`.

In [ ]:
#@title C · Tools: clips, label files, overlays  { display-mode: "form" }
# One format for everything, the same DAVIS layout Meta's SAM 2 training code reads:
#   dataset/JPEGImages/<clip>/00000.jpg ...      frames
#   dataset/Annotations/<clip>/00000.png ...     labels: pixel value = object id, 0 = background
# Model predictions are stored the same way under predictions/<model>/<clip>/, which is what
# lets one scoring function grade any model.

def _davis_palette():
    pal = np.zeros((256, 3), np.uint8)
    for i in range(256):
        c = i
        for j in range(8):
            pal[i, 0] |= ((c >> 0) & 1) << (7 - j)
            pal[i, 1] |= ((c >> 1) & 1) << (7 - j)
            pal[i, 2] |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
    return pal

PALETTE = _davis_palette()
# Overlay colours chosen to stand out against green leaves (no greens).
OVERLAY_COLORS = np.array([(255, 0, 255), (0, 255, 255), (255, 140, 0), (255, 235, 0), (255, 40, 40),
                           (40, 110, 255), (255, 150, 200), (255, 255, 255), (160, 60, 255),
                           (0, 190, 255), (255, 110, 90), (200, 170, 255)], np.uint8)

def save_idmap(path, idmap):
    """Write an instance-id image. Palette PNG up to 255 objects (DAVIS style), 16-bit PNG beyond."""
    idmap = np.asarray(idmap)
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if idmap.size == 0 or int(idmap.max()) <= 255:
        im = Image.frombytes("P", (idmap.shape[1], idmap.shape[0]), idmap.astype(np.uint8).tobytes())
        im.putpalette(PALETTE.flatten().tolist())
    else:
        im = Image.fromarray(idmap.astype(np.uint16))
    im.save(path)

def load_idmap(path):
    im = Image.open(path)
    if im.mode not in ("P", "L", "1", "I;16", "I"):
        raise ValueError(f"{path}: expected a PNG whose pixel values are object ids, got a {im.mode} image")
    return np.asarray(im).astype(np.uint16)

@dataclass
class Clip:
    name: str
    root: object = None                      # folder holding the frames folder; None = the dataset
    @property
    def dir(self): return (Path(self.root) if self.root else IMG) / self.name
    @functools.cached_property
    def frames(self): return sorted(self.dir.glob("*.jpg"))      # listed once per Clip object
    def __len__(self): return len(self.frames)
    @functools.cached_property
    def hw(self):
        w, h = Image.open(self.frames[0]).size
        return h, w
    def image(self, i): return np.asarray(Image.open(self.frames[i]).convert("RGB"))
    def labels(self, root=None):
        """All label frames as one [T, H, W] array, or None if the clip is not fully labelled."""
        d = (ANN if root is None else Path(root)) / self.name
        files = [d / (f.stem + ".png") for f in self.frames]
        if not files or not all(f.exists() for f in files):
            return None
        return np.stack([load_idmap(f) for f in files])
    def save_labels(self, idmaps, root=None, only=None):
        """Write one PNG per frame (files are overwritten in place). only = frame numbers to write."""
        d = (ANN if root is None else Path(root)) / self.name
        keep = {f.stem + ".png" for f in self.frames}
        for t, (f, m) in enumerate(zip(self.frames, idmaps)):
            if only is None or t in only:
                save_idmap(d / (f.stem + ".png"), m)
        for stale in d.glob("*.png"):
            if stale.name not in keep: stale.unlink()

# ---- the clip table (one small JSON next to the data keeps split + origin of every clip) ----
def _load_meta(): return json.loads(_META.read_text()) if _META.exists() else {}
def _save_meta(m): _META.write_text(json.dumps(m, indent=1))
def set_meta(clip, **kw):
    m = _load_meta(); m.setdefault(clip, {}).update(kw); _save_meta(m)

_count_cache = {}
def _count_objects(files):
    """Distinct object ids across label files; re-read only when a file has changed."""
    key = tuple((str(f), f.stat().st_mtime_ns, f.stat().st_size) for f in files)
    if _count_cache.get(files[0].parent if files else None, (None,))[0] != key:
        objs = set()
        for f in files: objs |= set(np.unique(load_idmap(f)).tolist())
        _count_cache[files[0].parent if files else None] = (key, len(objs - {0}))
    return _count_cache[files[0].parent if files else None][1]

def clips_table():
    """Every clip on disk. labels: none | draft (not yet approved by a person) | done (approved).
    Only 'done' clips are used for training and scoring."""
    meta, rows = _load_meta(), []
    for d in sorted(p for p in IMG.iterdir() if p.is_dir()):
        c = Clip(d.name); info = meta.get(c.name, {}); n = len(c)
        labs = sorted((ANN / c.name).glob("*.png")) if (ANN / c.name).exists() else []
        status = (info.get("status") or "draft") if (n and len(labs) == n) else "none"
        rows.append(dict(clip=c.name, split=info.get("split", ""), frames=n, labels=status,
                         labelled=status == "done", objects=_count_objects(labs) if labs else 0,
                         source=info.get("source", ""), start_s=info.get("start_s", "")))
    t = pd.DataFrame(rows, columns=["clip", "split", "frames", "labels", "labelled", "objects", "source", "start_s"])
    return t.astype({"labelled": bool, "frames": int, "objects": int})          # right types even when empty

def get_clips(split=None, labelled=None):
    t = clips_table()
    if split is not None: t = t[t["split"] == split]
    if labelled is not None: t = t[t["labelled"] == labelled]
    return [Clip(n) for n in t["clip"]]

# ---- drawing -----------------------------------------------------------------------------------
def overlay(img, idmap, alpha=0.45, ids=True, outline=True):
    """Colour each object on top of the frame. Returns an RGB uint8 image."""
    out = img.copy()
    present = np.unique(idmap); present = present[present != 0]
    if len(present) == 0:
        return out
    color = OVERLAY_COLORS[(idmap.astype(np.int64) - 1) % len(OVERLAY_COLORS)]
    m = idmap > 0
    out[m] = (out[m] * (1 - alpha) + color[m] * alpha).astype(np.uint8)
    if outline:
        k, f = np.ones((3, 3), np.uint8), idmap.astype(np.float32)
        edge = cv2.dilate(f, k) != cv2.erode(f, k)
        out[edge & m] = color[edge & m]          # solid object colour just inside the boundary
        out[edge & ~m] = 255                     # white line just outside it
    if ids:
        s = max(0.45, min(out.shape[:2]) / 900)
        ys, xs = np.nonzero(m); lab = idmap[ys, xs].astype(np.int64)
        cnt = np.bincount(lab); cy = np.bincount(lab, ys); cx = np.bincount(lab, xs)
        for k_ in present:
            p = (int(cx[k_] / cnt[k_]) - int(8 * s), int(cy[k_] / cnt[k_]) + int(8 * s))
            cv2.putText(out, str(int(k_)), p, cv2.FONT_HERSHEY_SIMPLEX, s, (0, 0, 0), max(2, int(4 * s)), cv2.LINE_AA)
            cv2.putText(out, str(int(k_)), p, cv2.FONT_HERSHEY_SIMPLEX, s, (255, 255, 255), max(1, int(1.5 * s)), cv2.LINE_AA)
    return out

def show_clip(clip, idmaps=None, n=4, title=None, width=16):
    """Contact sheet of n evenly spaced frames. idmaps defaults to the clip's saved labels."""
    clip = Clip(clip) if isinstance(clip, str) else clip
    if idmaps is None: idmaps = clip.labels()
    T = len(clip); picks = sorted(set(np.linspace(0, T - 1, min(n, T)).round().astype(int).tolist()))
    h, w = clip.hw
    fig, axes = plt.subplots(1, len(picks), figsize=(width, width / len(picks) * h / w + 0.4), squeeze=False)
    for ax, i in zip(axes[0], picks):
        im = clip.image(i)
        ax.imshow(overlay(im, idmaps[i]) if idmaps is not None else im)
        ax.set_title(f"frame {i}", fontsize=9); ax.axis("off")
    fig.suptitle(title or clip.name, fontsize=11); plt.tight_layout(); plt.show()

def write_video(frames, path, fps=6):
    """Write RGB frames (a list or a generator) to an .mp4 that plays in the browser (H.264 if ffmpeg is there)."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp, vw = path.with_suffix(".tmp.mp4"), None
    for f in frames:
        h, w = f.shape[0] - f.shape[0] % 2, f.shape[1] - f.shape[1] % 2
        if vw is None:
            vw = cv2.VideoWriter(str(tmp), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
        vw.write(cv2.cvtColor(np.ascontiguousarray(f[:h, :w]), cv2.COLOR_RGB2BGR))
    if vw is None:
        raise ValueError("no frames to write")
    vw.release()
    if shutil.which("ffmpeg") and subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(tmp), "-vcodec", "libx264", "-pix_fmt", "yuv420p", str(path)]).returncode == 0:
        tmp.unlink()
    else:
        tmp.replace(path)
    return path

# ---- videos → clips → train / val ---------------------------------------------------------------
VIDEO_EXTS = {".mp4", ".mov", ".m4v", ".avi", ".mkv", ".mts", ".webm"}

def probe_video(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return dict(fps=float(fps), frames=n, width=w, height=h, seconds=(n / fps if fps else 0.0))

def list_videos():
    rows = []
    for p in sorted(VIDEOS.iterdir()):
        if p.suffix.lower() in VIDEO_EXTS:
            info = probe_video(p)
            rows.append(dict(video=p.name, MB=round(p.stat().st_size / 2**20, 1), **(info or {"fps": None})))
    return pd.DataFrame(rows)

def cut_clips(video, clips_per_video=6, clip_frames=16, clip_fps=6.0, max_side=1280, starts=None):
    """Extract evenly spaced, non-overlapping clips from one video. Safe to re-run: existing clips are kept."""
    video = Path(video); info = probe_video(video)
    if not info or not info["fps"] or not info["frames"]:
        print(f"  ✗ cannot read {video.name}"); return []
    stride = max(1, round(info["fps"] / clip_fps))          # keep every stride-th frame
    span = clip_frames * stride / info["fps"]               # seconds of video one clip covers
    dur = info["seconds"]
    if starts is None:
        k = min(clips_per_video, max(1, int(dur // span)))
        starts = [(i + 0.5) * dur / k - span / 2 for i in range(k)]
    taken = {n: (m.get("start_s") or 0, (m.get("start_s") or 0) + m.get("span_s", 0)) for n, m in _load_meta().items()
             if m.get("source") == video.name and (IMG / n).exists()}
    made, cap = [], cv2.VideoCapture(str(video))
    for s in starts:
        s = max(0.0, min(s, max(0.0, dur - span)))
        name = f"{_safe(video.stem)}_t{int(round(s * 1000)):07d}"
        out = IMG / name
        if name in taken:
            made.append(name); continue
        clash = [n for n, (a, b) in taken.items() if s < b and a < s + span]
        if clash:                      # never create a clip that shares frames with an existing one
            print(f"  · {name} would overlap {clash[0]}, skipped"); continue
        cap.set(cv2.CAP_PROP_POS_MSEC, s * 1000)
        frames, i = [], 0
        while len(frames) < clip_frames and cap.grab():
            if i % stride == 0:
                ok, bgr = cap.retrieve()
                if not ok: break
                h, w = bgr.shape[:2]; sc = max_side / max(h, w)
                if sc < 1:
                    bgr = cv2.resize(bgr, (round(w * sc), round(h * sc)), interpolation=cv2.INTER_AREA)
                frames.append(bgr)
            i += 1
        if len(frames) < clip_frames:
            print(f"  ✗ {name}: only {len(frames)} of {clip_frames} frames readable, skipped"); continue
        out.mkdir(parents=True, exist_ok=True)
        for j, f in enumerate(frames):
            cv2.imwrite(str(out / f"{j:05d}.jpg"), f, [cv2.IMWRITE_JPEG_QUALITY, 95])
        set_meta(name, source=video.name, start_s=round(s, 3), span_s=round(span, 3),
                 fps=round(info["fps"] / stride, 2))
        taken[name] = (s, s + span); made.append(name)
    cap.release()
    return made

def split_clips(val_fraction=0.25, reset=False):
    """Give every clip without a split a 'train' or 'val' tag.
    3+ source videos: whole videos are held out. Fewer: the last clips (in time) of each video are held out.
    Clips that already have a split keep it, so adding videos later never moves a val clip into train."""
    meta = _load_meta()
    names = [d.name for d in sorted(IMG.iterdir()) if d.is_dir()]
    for n in names:
        meta.setdefault(n, {})
        if reset: meta[n]["split"] = ""
    by_src = {}
    for n in names:
        by_src.setdefault(meta[n].get("source") or n, []).append(n)
    count = lambda s: sum(1 for n in names if meta[n].get("split") == s)
    whole_videos = len(by_src) >= 3                 # a source = one video (clips cut here, or imported frames)
    for src in sorted(by_src):
        cl = sorted(by_src[src], key=lambda n: (meta[n].get("start_s") or 0, n))
        todo = [n for n in cl if not meta[n].get("split")]
        if not todo: continue
        already = {meta[n]["split"] for n in cl if meta[n].get("split")}
        if whole_videos and len(already) <= 1:
            s = already.pop() if already else (
                "val" if count("val") < val_fraction * (count("train") + count("val") + len(todo)) else "train")
            for n in todo: meta[n]["split"] = s
        else:
            k = max(1, round(len(cl) * val_fraction)) if len(cl) > 1 else 0
            for i, n in enumerate(cl):
                if n in todo: meta[n]["split"] = "val" if i >= len(cl) - k else "train"
    _save_meta(meta)
    return "whole videos held out" if whole_videos else "last clips of each video held out"

def set_split(clip, split):
    assert split in ("train", "val"); set_meta(clip, split=split)

def delete_clip(clip):
    for d in (IMG / clip, ANN / clip):
        if d.exists(): shutil.rmtree(d)
    m = _load_meta(); m.pop(clip, None); _save_meta(m)

VAL_FRACTION = 0.25          # share of clips (or of whole videos) held out for scoring; Step 2 can change it

def dataset_report(show_examples=True):
    """What is usable for training and scoring, plus the problems worth fixing first."""
    t = clips_table()
    display(t)
    ok = t.loc[t["labelled"].astype(bool)]
    n = {s: int((ok["split"] == s).sum()) for s in ("train", "val")}
    print(f"usable (labelled and marked done): {n['train']} train clip(s), {n['val']} val clip(s), "
          f"{int(ok['objects'].sum())} labelled objects in total")
    for msg, bad in (("no labelled val clips: nothing can be scored yet", n["val"] == 0),
                     ("no labelled train clips: nothing to fine-tune on yet", n["train"] == 0),
                     (f"{int((t['labels'] == 'draft').sum())} clip(s) are still drafts: open them in label() and mark them done",
                      bool((t["labels"] == "draft").any())),
                     ("fewer than 3 source videos: train and val clips come from the same video, so scores will flatter "
                      "every model. Film val footage on a different row or day when you can.",
                      0 < t["source"].nunique() < 3 and not DEMO),
                     ("under 10 labelled train clips is enough to test the plumbing, not to expect a real gain",
                      0 < n["train"] < 10 and not DEMO)):
        if bad: print("⚠️ ", msg)
    if show_examples and _in_notebook():
        for c in get_clips("train", labelled=True)[:1] + get_clips("val", labelled=True)[:1]:
            show_clip(c, title=f"{c.name}  ({_load_meta()[c.name].get('split')})")
    return t

# ---- public sample data (Embrapa WGISD, CC BY-NC 4.0): real grape photos with hand-drawn cluster masks -------
_WGISD = "https://raw.githubusercontent.com/thsant/wgisd/master/data/"
DEMO_IMAGES = ["CDY_2015", "CDY_2017", "CDY_2018", "CDY_2019", "SVB_1931", "SVB_1935", "SVB_1937", "SVB_1938"]

def make_demo_clips(names=None, frames=12, fps=6, hw=(640, 960)):
    """Turn WGISD photos into short panning clips with exact labels, through the same code path as real videos
    (written as .mp4 into videos/, cut by cut_clips). Gives every step something to run on before you have
    labelled anything. Returns the clip names; existing demo clips are kept."""
    raw = BASE / "demo_download"; raw.mkdir(parents=True, exist_ok=True)
    made = []
    for name in names or DEMO_IMAGES:
        clip_name = f"demo_{name}_t0000000"
        if (ANN / clip_name).exists():
            made.append(clip_name); continue
        for ext in ("jpg", "npz"):
            f = raw / f"{name}.{ext}"
            if not f.exists():
                torch.hub.download_url_to_file(_WGISD + f.name, str(f), progress=False)
        img = np.asarray(Image.open(raw / f"{name}.jpg").convert("RGB"))
        masks = np.load(raw / f"{name}.npz")["arr_0"]                        # [H, W, N], one layer per cluster
        ids = np.zeros(img.shape[:2], np.uint16)
        for k in np.argsort(-masks.reshape(-1, masks.shape[2]).sum(0)):      # large first, small ones on top
            ids[masks[..., k] > 0] = k + 1
        H, W = ids.shape; oh, ow = hw
        win_w = int(W * 0.6); win_h = int(win_w * oh / ow)
        clip_frames, labels = [], []
        for i in range(frames):                                              # pan left → right with a gentle bob
            a = i / (frames - 1)
            x0 = int(a * (W - win_w)); y0 = int((H - win_h) * (0.5 + 0.25 * math.sin(a * math.pi)))
            clip_frames.append(cv2.resize(img[y0:y0 + win_h, x0:x0 + win_w], (ow, oh), interpolation=cv2.INTER_AREA))
            labels.append(cv2.resize(ids[y0:y0 + win_h, x0:x0 + win_w], (ow, oh), interpolation=cv2.INTER_NEAREST))
        vid = VIDEOS / f"demo_{name}.mp4"
        vw = cv2.VideoWriter(str(vid), cv2.VideoWriter_fourcc(*"mp4v"), fps, (ow, oh))
        for f in clip_frames:
            vw.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
        vw.release()
        got = cut_clips(vid, 1, frames, fps, max(hw))                        # the same path your videos take
        assert got == [clip_name] and len(Clip(clip_name)) == frames, f"demo clip {name} did not extract cleanly"
        Clip(clip_name).save_labels(np.stack(labels))
        set_meta(clip_name, status="done")
        made.append(clip_name)
    return made

In [ ]:
#@title Tools: SAM 2.1 (load a model, turn hints into tracked masks)  { display-mode: "form" }
import contextlib, gc
import sam2.sam2_video_predictor as _svp, sam2.utils.misc as _misc
_svp.tqdm = _misc.tqdm = lambda it, **kw: it          # silence SAM 2's per-clip progress bars

SAM2_MODELS = {   # the four SAM 2.1 sizes (released 2024-09-29; the newest checkpoints in the SAM 2 line)
    "tiny":      dict(cfg="configs/sam2.1/sam2.1_hiera_t.yaml",  file="sam2.1_hiera_tiny.pt",      mirror="sam2.1_t.pt"),
    "small":     dict(cfg="configs/sam2.1/sam2.1_hiera_s.yaml",  file="sam2.1_hiera_small.pt",     mirror="sam2.1_s.pt"),
    "base_plus": dict(cfg="configs/sam2.1/sam2.1_hiera_b+.yaml", file="sam2.1_hiera_base_plus.pt", mirror="sam2.1_b.pt"),
    "large":     dict(cfg="configs/sam2.1/sam2.1_hiera_l.yaml",  file="sam2.1_hiera_large.pt",     mirror="sam2.1_l.pt"),
}
SAM2_IMAGE_SIZE = 1024   # advanced: 768 or 512 run faster and lighter but a little less accurately (training and inference alike)
_OFFICIAL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/"
_MIRROR = "https://github.com/ultralytics/assets/releases/download/v8.3.0/"

def sam2_checkpoint(size):
    """Path to the pretrained SAM 2.1 weights for `size`, downloading them once."""
    info = SAM2_MODELS[size]; dst = CKPT_DIR / info["file"]
    if dst.exists():
        return dst
    tmp = dst.with_suffix(".part")
    print(f"downloading SAM 2.1 {size} weights (once per session) …")
    try:
        torch.hub.download_url_to_file(_OFFICIAL + info["file"], str(tmp), progress=False)
    except Exception as e:      # Meta's server unreachable: fall back to the copy Ultralytics hosts on GitHub
        print(f"official download failed ({type(e).__name__}); using the GitHub mirror (half-precision copy of the same weights)")
        torch.hub.download_url_to_file(_MIRROR + info["mirror"], str(tmp), progress=False)
        sd = torch.load(tmp, map_location="cpu", weights_only=True)["model"]
        torch.save({"model": {k: v.float() for k, v in sd.items()}}, tmp)
    tmp.replace(dst)
    return dst

_SAM2_OVERRIDES = [   # same inference settings Meta's builder applies, minus hole filling (needs a compiled CUDA op)
    "++model.sam_mask_decoder_extra_args.dynamic_multimask_via_stability=true",
    "++model.sam_mask_decoder_extra_args.dynamic_multimask_stability_delta=0.05",
    "++model.sam_mask_decoder_extra_args.dynamic_multimask_stability_thresh=0.98",
    "++model.binarize_mask_from_pts_for_mem_enc=true",
]
_models, _info = {}, {}

def checkpoint_info(ckpt):
    """Size and resolution a fine-tuned checkpoint was trained at (written next to it by finetune())."""
    if ckpt is None:
        return {}
    key = (str(ckpt), Path(ckpt).stat().st_mtime_ns)
    if key not in _info:
        side = Path(ckpt).with_suffix(".json")
        if side.exists():
            _info[key] = json.loads(side.read_text())
        else:                                   # the same two facts are stored inside the .pt as a fallback
            extra = torch.load(ckpt, map_location="cpu", weights_only=True)
            _info[key] = {k: extra[k] for k in ("size", "image_size") if k in extra}
    return _info[key]

def load_sam2(size=None, ckpt=None):
    """A SAM 2.1 video predictor. ckpt=None → Meta's pretrained weights for `size`; or the path of a fine-tuned
    .pt file (its size and resolution are read from the file, so `size` can be left out)."""
    info = checkpoint_info(ckpt)
    size = info.get("size") or size or SAM2_SIZE
    side = int(info.get("image_size") or SAM2_IMAGE_SIZE)
    key = (size, str(ckpt), side)
    if key not in _models:
        from sam2.build_sam import build_sam2_video_predictor
        _models[key] = build_sam2_video_predictor(
            SAM2_MODELS[size]["cfg"], str(ckpt or sam2_checkpoint(size)), device=DEVICE, apply_postprocessing=False,
            hydra_overrides_extra=_SAM2_OVERRIDES + [f"++model.image_size={side}"])
    return _models[key]

def free_models():
    _models.clear(); gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

def _amp():
    return torch.autocast("cuda", dtype=torch.bfloat16) if DEVICE == "cuda" else contextlib.nullcontext()

class Sam2Session:
    """One clip loaded into SAM 2. `add()` tracks a set of objects and paints them into id maps.

    A prompt is a dict: {"obj": id, "frame": t} plus "box": [x0, y0, x1, y1] and/or "points": [[x, y], ..] with
    "labels": [1 = belongs, 0 = does not, ..], or "mask": bool[H, W]. An object may have prompts on several
    frames; each prompted frame is segmented from its own prompt and the frames in between are tracked."""

    def __init__(self, predictor, clip):
        self.p, self.clip, self.T = predictor, clip, len(clip)
        self.H, self.W = clip.hw
        with torch.inference_mode(), _amp():
            self.state = predictor.init_state(video_path=str(clip.dir))

    def _apply(self, pr):
        if pr.get("mask") is not None:
            self.p.add_new_mask(self.state, int(pr["frame"]), int(pr["obj"]), np.asarray(pr["mask"], bool))
            return
        pts = np.asarray(pr["points"], np.float32).reshape(-1, 2) if pr.get("points") else None
        self.p.add_new_points_or_box(
            self.state, int(pr["frame"]), int(pr["obj"]), points=pts,
            labels=np.asarray(pr["labels"], np.int32) if pts is not None else None,
            box=np.asarray(pr["box"], np.float32) if pr.get("box") is not None else None)

    @torch.inference_mode()
    def add(self, prompts, ids=None, best=None, backward=True):
        """Track every object in `prompts` and paint it into `ids` (the highest positive score owns a pixel).
        Objects whose first prompt is on the same frame are tracked together, each start frame in its own
        pass, the way Meta's VOS script handles objects that appear later. backward=False leaves the frames
        before an object's first prompt empty (the scoring protocol); backward=True fills them in (labelling)."""
        if ids is None: ids = np.zeros((self.T, self.H, self.W), np.uint16)
        if best is None: best = np.zeros((self.T, self.H, self.W), np.float16)
        per_obj = {}
        for pr in prompts: per_obj.setdefault(int(pr["obj"]), []).append(pr)
        groups = {}
        for o, ps in per_obj.items(): groups.setdefault(min(int(p["frame"]) for p in ps), []).append(o)
        with _amp():
            for f0 in sorted(groups):
                self.p.reset_state(self.state)
                for o in groups[f0]:
                    for pr in per_obj[o]: self._apply(pr)
                passes = [dict(start_frame_idx=f0)] + ([dict(start_frame_idx=f0, reverse=True)] if backward and f0 > 0 else [])
                for kw in passes:
                    for t, obj_ids, logits in self.p.propagate_in_video(self.state, **kw):
                        if kw.get("reverse") and t == f0:
                            continue
                        mx, arg = logits[:, 0].float().max(0)
                        mx, arg = mx.cpu().numpy(), arg.cpu().numpy()
                        win = mx > best[t]
                        ids[t][win] = np.asarray(obj_ids, np.uint16)[arg[win]]
                        best[t][win] = np.minimum(mx[win], 1000.0)
        return ids

def sam2_track(predictor, clip, prompts, backward=True):
    """Prompts → [T, H, W] array of object ids for the whole clip."""
    return Sam2Session(predictor, clip).add(prompts, backward=backward)

def prompts_from_labels(gt, kind="box", visible_frac=0.5, min_area=64):
    """Simulate the hint a person (or a detector) would give: one prompt per labelled object, on the first
    frame where at least `visible_frac` of its largest view is in frame. kind: 'box' | 'point' | 'mask'."""
    T = gt.shape[0]; top = int(gt.max())
    area = np.stack([np.bincount(gt[t].ravel(), minlength=top + 1) for t in range(T)])
    out = []
    for k in range(1, top + 1):
        a = area[:, k]
        if a.max() == 0: continue
        ok = a >= max(min_area, visible_frac * a.max())
        f = int(np.argmax(ok)) if ok.any() else int(a.argmax())
        m = gt[f] == k
        pr = dict(obj=k, frame=f)
        if kind == "mask":
            pr["mask"] = m
        elif kind == "box":
            ys, xs = np.nonzero(m)
            pr["box"] = [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]
        elif kind == "point":      # the pixel deepest inside the object, as in SAM's own click evaluation
            d = cv2.distanceTransform(np.pad(m, 1).astype(np.uint8), cv2.DIST_L2, 5)[1:-1, 1:-1]
            y, x = np.unravel_index(int(d.argmax()), d.shape)
            pr["points"], pr["labels"] = [[float(x), float(y)]], [1]
        else:
            raise ValueError(kind)
        out.append(pr)
    return out

In [ ]:
#@title Tools: labelling  { display-mode: "form" }
import copy, zlib, gc
import anywidget, traitlets, ipywidgets as widgets
if IN_COLAB:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()         # lets Colab show the drawing canvas


def _load_prompts(clip):
    f = PROMPTS / f"{clip}.json"
    return json.loads(f.read_text()) if f.exists() else []

def _save_prompts(clip, prompts):
    PROMPTS.mkdir(exist_ok=True)
    (PROMPTS / f"{clip}.json").write_text(json.dumps(prompts))

class ClipLabels:
    """The labels of one clip plus the operations to change them. No screen needed, so it also works from code:
        ed = ClipLabels("my_clip"); ed.add_box(0, [x0, y0, x1, y1]); ed.delete(3); ed.save(done=True)"""

    def __init__(self, clip, size=None, ckpt=None):
        self.clip = Clip(clip) if isinstance(clip, str) else clip
        self.T = len(self.clip); self.H, self.W = self.clip.hw
        lab = self.clip.labels()
        self.ids = lab if lab is not None else np.zeros((self.T, self.H, self.W), np.uint16)
        self.best = np.where(self.ids > 0, 1000, 0).astype(np.float16)   # existing labels keep their pixels
        self.prompts = _load_prompts(self.clip.name)
        self._model, self._sess, self._undo = (size, ckpt), None, []
        self._on_disk = self.ids.copy() if lab is not None else None       # what the label files currently hold
        self.status_now = (_load_meta().get(self.clip.name, {}).get("status") or "draft") if lab is not None else None

    @property
    def sess(self):
        if self._sess is None:
            self._sess = Sam2Session(load_sam2(self._model[0], self._model[1]), self.clip)
        return self._sess

    def objects(self):
        return sorted(set(np.unique(self.ids).tolist()) - {0})

    def _snapshot(self):
        self._undo = (self._undo + [(zlib.compress(self.ids.tobytes(), 1), zlib.compress(self.best.tobytes(), 1),
                                     copy.deepcopy(self.prompts))])[-20:]

    def _erase(self, obj):
        m = self.ids == obj
        self.ids[m] = 0; self.best[m] = 0

    def _retrack(self, obj):
        self._erase(obj)
        mine = [p for p in self.prompts if p["obj"] == obj]
        if mine:
            self.sess.add(mine, self.ids, self.best, backward=True)
        self.save()

    def _clean_box(self, box):
        x0, y0, x1, y1 = [float(v) for v in box]
        x0, x1 = sorted((min(max(x0, 0), self.W - 1), min(max(x1, 0), self.W - 1)))
        y0, y1 = sorted((min(max(y0, 0), self.H - 1), min(max(y1, 0), self.H - 1)))
        if x1 - x0 < 4 or y1 - y0 < 4:
            raise ValueError("that box is too small")
        return [x0, y0, x1, y1]

    def add_box(self, frame, box):
        """Draw a box around one object on one frame; SAM 2 fills in the mask on every frame. Returns the new id."""
        box = self._clean_box(box)
        new = max([0] + self.objects() + [p["obj"] for p in self.prompts]) + 1
        if new > 255:
            raise ValueError("255 objects per clip is the limit of the label format")
        self._snapshot()
        self.prompts.append(dict(obj=new, frame=int(frame), box=box))
        self._retrack(new)
        return new

    def set_box(self, obj, frame, box):
        """The reliable fix: say where an existing object is on this frame by drawing its box again. Replaces
        any earlier box or clicks for that object on that frame; the other frames are re-tracked from it."""
        obj, frame, box = int(obj), int(frame), self._clean_box(box)
        if obj not in self.objects() and not any(p["obj"] == obj for p in self.prompts):
            raise ValueError(f"there is no object {obj}")
        self._snapshot()          # (an imported or drafted object is re-made from this box on every frame)
        self.prompts = [p for p in self.prompts if not (p["obj"] == obj and p["frame"] == frame)]
        self.prompts.append(dict(obj=obj, frame=frame, box=box))
        self._retrack(obj)

    def add_click(self, obj, frame, x, y, positive=True):
        """Fix an object on one frame: a positive click says 'this belongs to it', a negative click 'this does
        not'. That frame is then re-segmented from the object's outline there plus all clicks on it, and the
        other frames are re-tracked."""
        obj, frame, x, y = int(obj), int(frame), float(x), float(y)
        if not any(p["obj"] == obj for p in self.prompts):
            raise ValueError(f"object {obj} was imported or drafted without a box, so clicks cannot nudge it: "
                             "use Re-box selected (set_box) instead")
        rec = next((p for p in self.prompts if p["obj"] == obj and p["frame"] == frame), None)
        ys, xs = np.nonzero(self.ids[frame] == obj)
        if rec is None and len(xs) == 0 and not positive:
            raise ValueError("the object is not on this frame, so there is nothing to remove")
        self._snapshot()
        if rec is None:                       # start from where the object currently is on this frame
            rec = dict(obj=obj, frame=frame, points=[], labels=[])
            if len(xs): rec["box"] = [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]
            self.prompts.append(rec)
        rec.setdefault("points", []).append([x, y]); rec.setdefault("labels", []).append(1 if positive else 0)
        if positive and rec.get("box"):       # a 'belongs' click outside the box stretches the box to reach it
            b, pad = rec["box"], 8.0
            rec["box"] = [max(min(b[0], x - pad), 0.0), max(min(b[1], y - pad), 0.0),
                          min(max(b[2], x + pad), self.W - 1.0), min(max(b[3], y + pad), self.H - 1.0)]
        self._retrack(obj)

    def delete(self, obj):
        self._snapshot()
        self._erase(obj)
        self.prompts = [p for p in self.prompts if p["obj"] != obj]
        self.save()

    def undo(self):
        if not self._undo:
            return False
        ids, best, self.prompts = self._undo.pop()
        self.ids = np.frombuffer(zlib.decompress(ids), np.uint16).reshape(self.ids.shape).copy()
        self.best = np.frombuffer(zlib.decompress(best), np.float16).reshape(self.best.shape).copy()
        self.save()
        return True

    def save(self, done=None):
        """Labels are written after every change (only the frames that changed). done=True marks the clip as
        checked by a person; until then it is a draft and is not used for training or scoring."""
        changed = None if self._on_disk is None else {t for t in range(self.T) if not np.array_equal(self.ids[t], self._on_disk[t])}
        self.clip.save_labels(self.ids, only=changed); self._on_disk = self.ids.copy()
        _save_prompts(self.clip.name, self.prompts)
        if done is not None or self.status_now is None:
            self.status_now = "done" if done else "draft"
            set_meta(self.clip.name, status=self.status_now)

    def render(self, frame, selected=None, width=1100):
        """The frame with labels, the boxes/clicks placed on it, and the selected object outlined in white."""
        out = overlay(self.clip.image(frame), self.ids[frame])
        if selected:
            cs, _ = cv2.findContours((self.ids[frame] == selected).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            cv2.drawContours(out, cs, -1, (0, 0, 0), 5); cv2.drawContours(out, cs, -1, (255, 255, 255), 2)
        for p in self.prompts:
            if p["frame"] != frame: continue
            if p.get("box"):
                x0, y0, x1, y1 = [int(v) for v in p["box"]]
                cv2.rectangle(out, (x0, y0), (x1, y1), (255, 255, 255), 1)
            for (x, y), l in zip(p.get("points") or [], p.get("labels") or []):
                cv2.circle(out, (int(x), int(y)), 6, (0, 0, 0), -1)
                cv2.circle(out, (int(x), int(y)), 4, (60, 255, 60) if l else (255, 60, 60), -1)
        scale = min(1.0, width / self.W)
        if scale < 1:
            out = cv2.resize(out, (round(self.W * scale), round(self.H * scale)), interpolation=cv2.INTER_AREA)
        return out, scale

class _Canvas(anywidget.AnyWidget):
    """Shows an image and reports mouse gestures: a drag (box) or a click, in image pixels."""
    _esm = """
    function render({ model, el }) {
      const wrap = document.createElement("div");
      wrap.style.cssText = "position:relative;display:inline-block;max-width:100%;line-height:0;cursor:crosshair;touch-action:none;user-select:none";
      const img = document.createElement("img");
      img.draggable = false;
      img.style.cssText = "max-width:100%;height:auto;display:block;-webkit-user-drag:none";
      const rect = document.createElement("div");
      rect.style.cssText = "position:absolute;border:2px dashed #fff;box-shadow:0 0 0 1px #000;display:none;pointer-events:none";
      wrap.appendChild(img); wrap.appendChild(rect); el.appendChild(wrap);
      const show = () => { const src = model.get("image"); if (src) img.src = src; };
      model.on("change:image", show); show();
      let start = null;
      const at = (e) => {
        const r = img.getBoundingClientRect();
        const cx = Math.min(Math.max(e.clientX - r.left, 0), r.width), cy = Math.min(Math.max(e.clientY - r.top, 0), r.height);
        return { cx, cy, x: cx * img.naturalWidth / r.width, y: cy * img.naturalHeight / r.height };
      };
      wrap.addEventListener("pointerdown", (e) => {
        if (e.button !== 0) return;
        e.preventDefault(); start = at(e);
        try { wrap.setPointerCapture(e.pointerId); } catch (err) {}
      });
      wrap.addEventListener("pointermove", (e) => {
        if (!start) return;
        const p = at(e);
        rect.style.left = Math.min(start.cx, p.cx) + "px"; rect.style.top = Math.min(start.cy, p.cy) + "px";
        rect.style.width = Math.abs(p.cx - start.cx) + "px"; rect.style.height = Math.abs(p.cy - start.cy) + "px";
        rect.style.display = "block";
      });
      wrap.addEventListener("pointerup", (e) => {
        if (!start) return;
        const p = at(e), s = start; start = null; rect.style.display = "none";
        const drag = Math.abs(p.cx - s.cx) > 6 && Math.abs(p.cy - s.cy) > 6;
        const ev = drag ? { kind: "box", x0: Math.min(s.x, p.x), y0: Math.min(s.y, p.y), x1: Math.max(s.x, p.x), y1: Math.max(s.y, p.y) }
                        : { kind: "click", x: p.x, y: p.y };
        ev.stamp = Date.now() + Math.random();
        model.set("event", ev); model.save_changes();
      });
      wrap.addEventListener("pointercancel", () => { start = null; rect.style.display = "none"; });
    }
    export default { render };
    """
    image = traitlets.Unicode("").tag(sync=True)
    event = traitlets.Dict({}).tag(sync=True)

class LabelEditor(ClipLabels):
    """ClipLabels with a point-and-click screen on top."""

    def __init__(self, clip, size=None, ckpt=None, on_status=None):
        super().__init__(clip, size, ckpt)
        self.selected, self._scale, self._on_status = None, 1.0, on_status
        self.canvas = _Canvas()
        self.slider = widgets.IntSlider(value=0, min=0, max=max(self.T - 1, 0), description="frame", continuous_update=False,
                                  readout=False, layout=widgets.Layout(width="420px"))
        self.prev, self.next = widgets.Button(description="◀", layout=widgets.Layout(width="40px")), widgets.Button(description="▶", layout=widgets.Layout(width="40px"))
        self.mode = widgets.ToggleButtons(options=[("Select", "select"), ("Add to selected", "pos"), ("Remove from selected", "neg")],
                                    value="select", description="a click will:", style={"button_width": "190px"})
        self.b_del, self.b_undo = widgets.Button(description="Delete selected", icon="trash"), widgets.Button(description="Undo", icon="undo")
        self.b_rebox = widgets.Button(description="Re-box selected", icon="crop", tooltip="then drag the correct box for the selected object on this frame")
        self._rebox = False
        self.b_done = widgets.Button(description="Clip checked: mark done", button_style="success", icon="check",
                               layout=widgets.Layout(width="230px"))
        self.status = widgets.HTML()
        self.slider.observe(lambda c: self.refresh(), names="value")
        self.prev.on_click(lambda b: setattr(self.slider, "value", max(self.slider.value - 1, 0)))
        self.next.on_click(lambda b: setattr(self.slider, "value", min(self.slider.value + 1, self.T - 1)))
        self.b_del.on_click(lambda b: self._do(lambda: self.delete(self.selected) if self.selected else None, clear=True))
        self.b_undo.on_click(lambda b: self._do(self.undo, clear=True))
        self.b_done.on_click(lambda b: self._do(self._approve))
        self.b_rebox.on_click(self._arm_rebox)
        self.canvas.observe(self._on_event, names="event")
        self.view = widgets.VBox([widgets.HBox([self.prev, self.slider, self.next, self.b_done]),
                            widgets.HBox([self.mode, self.b_rebox, self.b_del, self.b_undo]), self.status, self.canvas])
        self.refresh()

    def refresh(self, note=""):
        t = self.slider.value
        img, self._scale = self.render(t, self.selected)
        ok, buf = cv2.imencode(".jpg", cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 82])
        self.canvas.image = "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode()
        state = self.status_now or "no labels yet"
        sel = f" · selected <b>#{self.selected}</b>" if self.selected else ""
        self.status.value = (f"<b>{self.clip.name}</b> · frame {t + 1}/{self.T} · {len(self.objects())} object(s) · "
                             f"labels: <b>{state}</b>{sel}" + (f" · {note}" if note else ""))

    def _approve(self):
        self.save(done=True)
        if self._on_status: self._on_status()

    def close(self):
        """Release the clip's frames from the GPU when another clip is opened."""
        self._sess = None
        self.view.close(); gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()

    def _arm_rebox(self, b=None):
        self._rebox = bool(self.selected)
        self.refresh(note=f"now drag the correct box for <b>#{self.selected}</b> on this frame" if self.selected
                     else "click an object in Select mode first")

    def _do(self, action, clear=False):
        """Run an edit, show progress, and never let an error kill the screen."""
        try:
            self.status.value = "⏳ working…"
            action()
            if clear: self.selected = None
            self.refresh()
        except Exception as e:
            self.refresh(note=f"<span style='color:#c00'>{type(e).__name__}: {e}</span>")

    def _on_event(self, change):
        ev = change["new"]
        if not ev: return
        t, s = self.slider.value, self._scale
        if ev["kind"] == "box":
            box = [ev["x0"] / s, ev["y0"] / s, ev["x1"] / s, ev["y1"] / s]
            if self._rebox and self.selected:
                self._rebox = False
                self._do(lambda: self.set_box(self.selected, t, box))
            else:
                self._do(lambda: self.add_box(t, box))
        else:
            x, y = min(int(ev["x"] / s), self.W - 1), min(int(ev["y"] / s), self.H - 1)
            if self.mode.value == "select":
                self.selected = int(self.ids[t, y, x]) or None
                self.refresh()
            elif not self.selected:
                self.refresh(note="click an object in Select mode first")
            else:
                self._do(lambda: self.add_click(self.selected, t, x, y, positive=self.mode.value == "pos"))

def label(clip=None, size=None, ckpt=None):
    """Open the labelling screen. Pick a clip from the menu, or pass a clip name."""
    clip = clip.name if isinstance(clip, Clip) else clip
    names = [c for c in clips_table()["clip"]]
    if not names:
        print("No clips yet: add videos and cut clips first (Steps 1 and 2)."); return
    def menu():
        t = clips_table().set_index("clip")
        mark = {"done": "✅ done ", "draft": "✏️ draft", "none": "⬜ to do"}
        return [(f"{mark[t.loc[n, 'labels']]} · {t.loc[n, 'split'] or '?'} · {n}", n) for n in names]
    pick = widgets.Dropdown(options=menu(), value=clip or names[0], description="clip", layout=widgets.Layout(width="620px"))
    box, current = widgets.VBox([pick, widgets.HTML("loading…")]), [None]
    def refresh_menu():
        pick.unobserve(open_clip, names="value")
        keep = pick.value; pick.options = menu(); pick.value = keep
        pick.observe(open_clip, names="value")
    def open_clip(change=None):
        if current[0] is not None: current[0].close()
        refresh_menu()
        current[0] = LabelEditor(pick.value, size, ckpt, on_status=refresh_menu)
        box.children = (pick, current[0].view)
    pick.observe(open_clip, names="value")
    display(box)
    open_clip()

def draft_labels(model, clips=None, overwrite=False):
    """Let an automatic model take the first pass. Drafts are not used for training or scoring until a person
    opens them in label(), fixes what is wrong and presses 'mark done'."""
    t = clips_table()             # never touches clips a person has already approved
    todo = clips or [Clip(n) for n, lab in zip(t["clip"], t["labels"]) if lab == "none" or (overwrite and lab == "draft")]
    model.prepare()
    for clip in tqdm(todo, desc="drafting"):
        back = getattr(model, "backward", None)
        if back is not None: model.backward = True          # also label frames before the first detection
        try:
            ids = np.asarray(model.predict(clip))
        finally:
            if back is not None: model.backward = back
        clip.save_labels(ids)
        _save_prompts(clip.name, getattr(model, "last_prompts", []))
        set_meta(clip.name, status="draft")
    print(f"drafted {len(todo)} clip(s). Next: label() to check each one.")

def mark_done(clips):
    """Approve clips from code (for labels you have already checked some other way)."""
    for c in clips: set_meta(c if isinstance(c, str) else c.name, status="done")

def ruler(clip, frame=0, step=100):
    """No-screen fallback: show a frame with a pixel grid so box corners can be read off for ClipLabels.add_box."""
    c = Clip(clip) if isinstance(clip, str) else clip
    lab = c.labels()
    img = overlay(c.image(frame), lab[frame]) if lab is not None else c.image(frame)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(16, 16 * h / w))
    ax.imshow(img); ax.set_xticks(range(0, w, step)); ax.set_yticks(range(0, h, step))
    ax.grid(color="white", linewidth=0.6, alpha=0.7); ax.tick_params(labelsize=7, labeltop=True, labelright=True)
    plt.show()

In [ ]:
#@title Tools: labels in from Roboflow / CVAT / Label Studio, labels out to YOLO  { display-mode: "form" }
def _guess_source(filename, fallback):
    """Which video a labelled frame came from, read off its file name, so frames of one video never end up on
    both sides of the train / val split. Handles Roboflow ('clip_mp4-0012_jpg.rf.3f2a….jpg'), Label Studio
    ('1a2b3c4d-clip_0012.jpg') and '<video>_000123' / '<video>-frame-12' names. Bare numbers → `fallback`."""
    stem = re.sub(r"\.rf\.[0-9a-fA-F]+", "", Path(filename).name)
    stem = re.sub(r"_(jpe?g|png|bmp)$", "", Path(stem).stem, flags=re.I)
    stem = re.sub(r"^[0-9a-f]{8}-", "", stem)
    src = re.sub(r"[-_ .]*(frame|img|image)?[-_ .]*\d+$", "", stem, flags=re.I)
    src = re.sub(r"[_.-](mp4|mov|m4v|avi|mkv|mts|webm)$", "", src, flags=re.I)
    return _safe(src) or fallback

def _add_image_clip(name, img, ids, split, source, max_side, boxes=()):
    """Save one labelled image as a one-frame clip. `boxes` (x0, y0, x1, y1) are objects that came without an
    outline: SAM 2.1 draws their masks, and the clip stays a draft until a person checks it in label()."""
    h, w = ids.shape; sc = min(1.0, max_side / max(h, w))
    if sc < 1:
        size = (round(w * sc), round(h * sc))
        img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
        ids = cv2.resize(ids, size, interpolation=cv2.INTER_NEAREST)
        boxes = [[v * sc for v in b] for b in boxes]
    (IMG / name).mkdir(parents=True, exist_ok=True)
    Image.fromarray(img).save(IMG / name / "00000.jpg", quality=95)
    meta = dict(source=source, status="done")
    if split in ("train", "val"):
        meta["split"] = split
    if len(boxes):
        k0 = int(ids.max())
        prompts = [dict(obj=k0 + i, frame=0, box=[float(v) for v in b]) for i, b in enumerate(boxes, 1)]
        best = np.where(ids > 0, 1000, 0).astype(np.float16)[None]        # outlined objects keep their pixels
        ids = Sam2Session(load_sam2(), Clip(name)).add(prompts, ids[None].astype(np.uint16), best)[0]
        _save_prompts(name, prompts)              # lets label() refine them with clicks or a new box
        meta["status"] = "draft"
    save_idmap(ANN / name / "00000.png", ids)
    set_meta(name, **meta)
    return meta["status"]

def _finish_import(source, made, split, drafted, extra=""):
    if split == "auto":
        print("split:", split_clips(VAL_FRACTION))
    print(f"{source}: {made} image(s) imported" + (f" as '{split}'" if split != "auto" else "") + extra)
    if drafted:
        print(f"  {drafted} image(s) had boxes only: SAM 2.1 drew those masks, so they are drafts. Check them in "
              "label() and mark each one done, or accept them unchecked: mark_done([...]) in the notebook, "
              "`python fruit.py approve --all-drafts` on the command line.")

def import_coco(json_path, images_dir=None, split="auto", categories=None, source=None, max_side=1920):
    """COCO segmentation export (Roboflow 'COCO Segmentation', CVAT 'COCO 1.0', Label Studio 'COCO').
    Every image becomes a one-frame clip. `categories`: names to keep (None = all, each instance one object).
    split="auto" groups images by the video they came from (read from the file name) and holds whole videos
    out for val. Do not reuse Roboflow's own train/valid split: it mixes frames of one video across both.
    Objects with a box and no outline get a SAM 2.1 mask and leave the image as a draft to check."""
    json_path = Path(json_path); images_dir = Path(images_dir or json_path.parent)
    coco = json.loads(json_path.read_text())
    names = {c["id"]: c["name"] for c in coco["categories"]}
    keep = {i for i, n in names.items() if categories is None or n in categories}
    per_image = {}
    for a in coco["annotations"]:
        if a["category_id"] in keep and not a.get("iscrowd", 0):
            per_image.setdefault(a["image_id"], []).append(a)
    source = source or json_path.parent.name
    made = empty = missing = drafted = 0
    for im in tqdm(coco["images"], desc=f"import {source}"):
        src = images_dir / im["file_name"]
        if not src.exists():
            missing += 1; continue
        img = np.asarray(Image.open(src).convert("RGB")); h, w = img.shape[:2]
        sx, sy = w / im.get("width", w), h / im.get("height", h)
        anns = sorted(per_image.get(im["id"], []), key=lambda a: -a.get("area", 0))     # big first, small on top
        if len(anns) > 255:
            print(f"  {im['file_name']}: {len(anns)} objects, keeping the 255 largest"); anns = anns[:255]
        ids, boxes, k = np.zeros((h, w), np.uint16), [], 0
        for a in anns:
            seg = a.get("segmentation")
            if isinstance(seg, list) and any(len(p) >= 6 for p in seg):     # polygons
                polys = [(np.asarray(p, np.float32).reshape(-1, 2) * (sx, sy)).round().astype(np.int32) for p in seg if len(p) >= 6]
                m = np.zeros((h, w), np.uint8); cv2.fillPoly(m, polys, 1)
            elif isinstance(seg, dict):                                       # run-length masks
                from pycocotools import mask as _mu
                rle = _mu.frPyObjects(seg, seg["size"][0], seg["size"][1]) if isinstance(seg["counts"], list) else seg
                m = cv2.resize(_mu.decode(rle), (w, h), interpolation=cv2.INTER_NEAREST)
            else:                                                             # box only → SAM 2.1 draws it
                x, y, bw, bh = a["bbox"]
                boxes.append([x * sx, y * sy, (x + bw) * sx, (y + bh) * sy]); continue
            k += 1; ids[m > 0] = k
        empty += int(k == 0 and not boxes)
        stem = Path(im["file_name"]).stem
        status = _add_image_clip(_safe(f"{source}_{stem}")[:90], img, ids, split, _guess_source(im["file_name"], source),
                                 max_side, boxes)
        made += 1; drafted += status == "draft"
    _finish_import(source, made, split, drafted, (f", {empty} with no objects" if empty else "")
                   + (f", {missing} listed but not found in {images_dir}" if missing else ""))

def import_yolo_seg(images_dir, labels_dir, split="auto", classes=None, source=None, max_side=1920):
    """YOLO labels, one .txt per image. Outline lines ('class x1 y1 x2 y2 …', 0-1 coordinates) become masks;
    box lines ('class cx cy w h') get a SAM 2.1 mask and leave the image as a draft to check.
    `classes`: class numbers to keep (None = all). split="auto": see import_coco."""
    images_dir, labels_dir = Path(images_dir), Path(labels_dir)
    source = source or images_dir.parent.name or images_dir.name
    made = drafted = 0
    for src in tqdm(sorted(p for p in images_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")), desc=f"import {source}"):
        txt = labels_dir / (src.stem + ".txt")
        img = np.asarray(Image.open(src).convert("RGB")); h, w = img.shape[:2]
        shapes, boxes = [], []
        for line in (txt.read_text().splitlines() if txt.exists() else []):
            v = line.split()
            if len(v) < 5 or (classes is not None and int(float(v[0])) not in classes): continue
            c = np.asarray(v[1:], np.float32)
            if len(c) == 4:
                cx, cy, bw, bh = c
                boxes.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
            else:
                shapes.append((c[: len(c) // 2 * 2].reshape(-1, 2) * (w, h)).round().astype(np.int32))
        shapes.sort(key=lambda p: -cv2.contourArea(p))
        ids = np.zeros((h, w), np.uint16)
        for k, poly in enumerate(shapes[:255], 1):
            m = np.zeros((h, w), np.uint8); cv2.fillPoly(m, [poly], 1); ids[m > 0] = k
        boxes = boxes[: max(0, 255 - min(len(shapes), 255))]
        status = _add_image_clip(_safe(f"{source}_{src.stem}")[:90], img, ids, split, _guess_source(src.name, source),
                                 max_side, boxes)
        made += 1; drafted += status == "draft"
    _finish_import(source, made, split, drafted)

def _one_polygon(mask):
    """Outline of a mask as ONE polygon, which is what YOLO labels need. An object cut into several pieces
    (by a leaf, a post, another cluster) keeps all of them: the pieces are joined by zero-width bridges."""
    cs, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    parts = sorted((c.reshape(-1, 2) for c in cs if len(c) >= 3 and cv2.contourArea(c) >= 20),
                   key=lambda q: -cv2.contourArea(q))
    if not parts:
        return None
    poly = parts[0]
    for q in parts[1:]:
        d = ((poly[:, None, :].astype(np.int64) - q[None, :, :]) ** 2).sum(-1)
        i, j = np.unravel_index(int(d.argmin()), d.shape)        # closest pair of points: bridge there and back
        poly = np.concatenate([poly[:i + 1], q[j:], q[:j + 1], poly[i:]])
    return poly

def export_yolo_seg(out_dir=None, every=2, name=None):
    """Write the labelled clips as a YOLO segmentation dataset (same train / val split), ready for
    `YOLO('yolo26n-seg.pt').train(data=<returned yaml>)`. `every`: keep every n-th frame (neighbours are near-copies)."""
    out = Path(out_dir or ROOT / "yolo_dataset"); n = {"train": 0, "val": 0}
    for split in n:
        for sub in ("images", "labels"):
            shutil.rmtree(out / sub / split, ignore_errors=True); (out / sub / split).mkdir(parents=True)
        for clip in get_clips(split, labelled=True):
            lab = clip.labels(); h, w = clip.hw
            for t in range(0, len(clip), every):
                lines = []
                for k in np.unique(lab[t]):
                    if k == 0: continue
                    poly = _one_polygon(lab[t] == k)
                    if poly is None: continue
                    pts = (poly / (w, h)).clip(0, 1)
                    lines.append("0 " + " ".join(f"{v:.5f}" for v in pts.ravel()))
                stem = f"{clip.name}_{t:05d}"
                shutil.copy(clip.frames[t], out / "images" / split / f"{stem}.jpg")
                (out / "labels" / split / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
                n[split] += 1
    yaml = out / "data.yaml"
    yaml.write_text(f"path: {out}\ntrain: images/train\nval: images/val\nnames:\n  0: {name or OBJECT_NAME}\n")
    print(f"YOLO dataset: {n['train']} train + {n['val']} val images → {yaml}")
    return yaml

In [ ]:
#@title Tools: scoring (one yardstick for every model)  { display-mode: "form" }
# Two kinds of model, two leaderboards:
#   prompted  – the model is shown each object once (a box, a click or a mask) and must follow it.
#               Score: J&F, the standard video-segmentation metric (J = overlap, F = edge accuracy).
#   automatic – the model gets only pixels and must find every object itself.
#               Score: precision / recall / F1 at IoU 0.5, mask quality, and how far off the count is.
from scipy.optimize import linear_sum_assignment
from scipy import ndimage

def mask_iou(a, b):
    union = np.logical_or(a, b).sum()
    return 1.0 if union == 0 else float(np.logical_and(a, b).sum() / union)

def _boundary(seg):
    """1-pixel-wide outline of a binary mask (same definition as DAVIS and Meta's SA-V evaluator)."""
    e, s, se = np.zeros_like(seg), np.zeros_like(seg), np.zeros_like(seg)
    e[:, :-1] = seg[:, 1:]; s[:-1, :] = seg[1:, :]; se[:-1, :-1] = seg[1:, 1:]
    b = seg ^ e | seg ^ s | seg ^ se
    b[-1, :] = seg[-1, :] ^ e[-1, :]; b[:, -1] = seg[:, -1] ^ s[:, -1]; b[-1, -1] = False
    return b

def boundary_f(pred, gt, tolerance=0.008):
    """Boundary F-measure: do the two outlines agree to within 0.8% of the image diagonal?"""
    H, W = gt.shape
    r = int(np.ceil(tolerance * np.linalg.norm((H, W))))
    both = pred | gt
    if not both.any():
        return 1.0
    rows, cols = np.flatnonzero(both.any(1)), np.flatnonzero(both.any(0))       # crop: same answer, far faster
    y0, y1 = max(rows[0] - r - 2, 0), min(rows[-1] + r + 3, H)
    x0, x1 = max(cols[0] - r - 2, 0), min(cols[-1] + r + 3, W)
    pb, gb = _boundary(pred[y0:y1, x0:x1]), _boundary(gt[y0:y1, x0:x1])
    yy, xx = np.ogrid[-r:r + 1, -r:r + 1]
    disk = (xx * xx + yy * yy <= r * r).astype(np.uint8)
    n_p, n_g = int(pb.sum()), int(gb.sum())
    if n_p == 0 or n_g == 0:
        return 1.0 if n_p == n_g else 0.0
    precision = (pb & (cv2.dilate(gb.astype(np.uint8), disk) > 0)).sum() / n_p
    recall = (gb & (cv2.dilate(pb.astype(np.uint8), disk) > 0)).sum() / n_g
    return 0.0 if precision + recall == 0 else float(2 * precision * recall / (precision + recall))

def score_prompted(gt, pred, prompts):
    """One row per object, judged on the frames after its prompt.
    J, F          – averaged over frames where the object is in view (in the labels or in the prediction).
                    This is the headline: in a pan most objects leave the frame, and agreeing that an object
                    is gone is easy.
    J_all, F_all  – averaged over every frame, an empty-vs-empty frame counting as perfect. This is the
                    published DAVIS / SA-V convention, so it is comparable with papers but reads higher.
    iou_prompt_frame – mask quality on the very frame the model was pointed at the object."""
    T, rows = gt.shape[0], []
    for pr in prompts:
        k, f0 = pr["obj"], pr["frame"]
        J, F, seen = [], [], []
        for t in range(f0 + 1, T):
            g, p = gt[t] == k, pred[t] == k
            J.append(mask_iou(p, g)); F.append(boundary_f(p, g)); seen.append(bool(g.any() or p.any()))
        J, F, seen = np.array(J, float), np.array(F, float), np.array(seen, bool)
        avg = lambda v: 100 * float(v.mean()) if len(v) else np.nan
        rows.append(dict(obj=k, prompt_frame=f0, frames=len(J), frames_in_view=int(seen.sum()),
                         J=avg(J[seen]), F=avg(F[seen]), J_all=avg(J), F_all=avg(F),
                         iou_prompt_frame=100 * mask_iou(pred[f0] == k, gt[f0] == k)))
    return rows

def iou_matrix(a, b):
    """IoU between every object in id map `a` and every object in id map `b` (same frame).
    Returns (ids_a, ids_b, iou [len(ids_a), len(ids_b)], areas_a, areas_b); background is left out."""
    a_ids, a_inv = np.unique(a, return_inverse=True)
    b_ids, b_inv = np.unique(b, return_inverse=True)
    joint = np.bincount(a_inv.ravel() * len(b_ids) + b_inv.ravel(),
                        minlength=len(a_ids) * len(b_ids)).reshape(len(a_ids), len(b_ids))
    a_area, b_area = joint.sum(1), joint.sum(0)
    ka, kb = a_ids != 0, b_ids != 0
    inter = joint[ka][:, kb]
    iou = inter / np.maximum(a_area[ka][:, None] + b_area[kb][None, :] - inter, 1)
    return a_ids[ka], b_ids[kb], iou, a_area[ka], b_area[kb]

def match_instances(gt, pred, iou_thr=0.5, min_area=0):
    """One frame, no hints: pair predicted objects with labelled ones (best one-to-one pairing by IoU).
    Returns (true positives, false positives, misses, [IoU of each true positive], n_gt, n_pred)."""
    _, _, iou, g_area, p_area = iou_matrix(gt, pred)
    iou = iou[g_area >= min_area][:, p_area >= min_area]
    n_g, n_p = iou.shape
    matched = []
    if n_g and n_p:
        r, c = linear_sum_assignment(-iou)
        matched = [float(v) for v in iou[r, c] if v >= iou_thr]
    return len(matched), n_p - len(matched), n_g - len(matched), matched, n_g, n_p

def score_automatic(gt, pred, iou_thr=0.5, min_area=0):
    """One row per frame for a model that had to find the objects by itself."""
    rows = []
    for t in range(gt.shape[0]):
        tp, fp, fn, ious, n_g, n_p = match_instances(gt[t], pred[t], iou_thr, min_area)
        rows.append(dict(frame=t, tp=tp, fp=fp, fn=fn, iou_sum=sum(ious), n_gt=n_g, n_pred=n_p))
    return rows

In [ ]:
#@title Tools: model wrappers and the leaderboard  { display-mode: "form" }
# A model is anything with predict(clip, prompts) → [T, H, W] array of object ids (0 = background).
# To add your own: subclass Model, or write PNGs with another program and use FromFolder.

class Model:
    needs_prompts = False   # True → scored on the prompted leaderboard; False → on the automatic one
    tracks = False          # True → the same object keeps the same id from frame to frame
    prompt = None           # for prompted models: "box" | "point" | "mask"
    def prepare(self):      # load weights here so loading time is not counted as inference time
        pass
    def predict(self, clip, prompts=None):
        raise NotImplementedError

class SAM2Prompted(Model):
    """SAM 2.1 pointed at each object once. ckpt=None → Meta's pretrained weights; or a fine-tuned .pt file."""
    needs_prompts, tracks = True, True
    def __init__(self, size=None, ckpt=None, prompt="box"):
        self.size, self.ckpt, self.prompt = size or SAM2_SIZE, ckpt, prompt
    def prepare(self):
        load_sam2(self.size, self.ckpt)
    def predict(self, clip, prompts=None):
        return sam2_track(load_sam2(self.size, self.ckpt), clip, prompts, backward=False)

def _ultralytics(text=False):
    """Import ultralytics, installing it (and its tracker / text-prompt extras) on first use."""
    need = [pkg for mod, pkg in (("ultralytics", "ultralytics"), ("lap", "lap>=0.5.12")) if _missing(mod)]
    if text and _missing("clip"):
        need.append("git+https://github.com/ultralytics/CLIP.git")          # text encoder for YOLOE prompts
    if need:
        print("installing:", ", ".join(need))
        sh([sys.executable, "-m", "pip", "install", "-q", *need])
    threads = torch.get_num_threads()
    import ultralytics
    torch.set_num_threads(threads)                # importing ultralytics lowers the CPU thread count; put it back
    return ultralytics

class UltralyticsModel(Model):
    """Any Ultralytics weights: a YOLO-seg model you trained, or YOLOE with a text prompt (text='grape cluster').
    Used on its own it is an automatic model (ByteTrack ids); it can also feed boxes to SAM 2 (DetectThenSAM2)."""
    tracks = True
    def __init__(self, weights, text=None, conf=0.25, imgsz=1280, tracker="bytetrack.yaml"):
        self.weights, self.text, self.conf, self.imgsz, self.tracker = str(weights), text, conf, imgsz, tracker
        self._m = None
    @property
    def model(self):
        if self._m is None:
            ul = _ultralytics(text=bool(self.text))
            print(f"loading {Path(self.weights).name} (downloaded on first use) …")
            with contextlib.redirect_stdout(io.StringIO()):          # hides the download progress bars
                if self.text:
                    names = [self.text] if isinstance(self.text, str) else list(self.text)
                    self._m = ul.YOLOE(self.weights)
                    self._m.set_classes(names, self._m.get_text_pe(names))
                else:
                    self._m = ul.YOLO(self.weights)
        return self._m
    def prepare(self):
        self.model
    def _tracker_file(self):
        """ByteTrack only starts a track above its own score bar (0.25 by default), which would silently
        discard every detection between `conf` and 0.25. Tie the tracker's bars to `conf` instead."""
        import yaml, ultralytics
        base = Path(ultralytics.__file__).parent / "cfg" / "trackers" / self.tracker
        cfg = yaml.safe_load((base if base.exists() else Path(self.tracker)).read_text())
        cfg.update(track_high_thresh=float(self.conf), new_track_thresh=float(self.conf),
                   track_low_thresh=float(min(cfg.get("track_low_thresh", 0.1), self.conf / 2)))
        f = CKPT_DIR / f"{Path(self.tracker).stem}_conf{self.conf:g}.yaml"
        f.write_text(yaml.safe_dump(cfg))
        return str(f)
    def _unpack(self, r, hw):
        n = 0 if r.boxes is None else len(r.boxes)
        boxes = r.boxes.xyxy.cpu().numpy() if n else np.zeros((0, 4), np.float32)
        scores = r.boxes.conf.cpu().numpy() if n else np.zeros(0, np.float32)
        ids = r.boxes.id.int().cpu().numpy() if n and r.boxes.id is not None else None
        if n and r.masks is not None:
            masks = r.masks.data.cpu().numpy() > 0.5
            if masks.shape[1:] != tuple(hw):
                masks = np.stack([cv2.resize(m.astype(np.uint8), hw[::-1], interpolation=cv2.INTER_NEAREST) > 0 for m in masks])
        else:                                       # box-only weights: fall back to filled boxes
            masks = np.zeros((n, *hw), bool)
            for i, (x0, y0, x1, y1) in enumerate(boxes.round().astype(int)):
                masks[i, max(y0, 0):y1, max(x0, 0):x1] = True
        return boxes, scores, masks, ids
    def detect(self, img):
        """One RGB frame → (boxes [N,4] x0,y0,x1,y1, scores [N], masks [N,H,W])."""
        r = self.model.predict(img[..., ::-1], conf=self.conf, imgsz=self.imgsz, retina_masks=True, verbose=False)[0]
        return self._unpack(r, img.shape[:2])[:3]
    def predict(self, clip, prompts=None):
        T = len(clip); H, W = clip.hw
        out, remap, tracker = np.zeros((T, H, W), np.uint16), {}, self._tracker_file()
        for t in range(T):
            r = self.model.track(clip.image(t)[..., ::-1], persist=t > 0, conf=self.conf, imgsz=self.imgsz,
                                 tracker=tracker, retina_masks=True, verbose=False)[0]
            boxes, scores, masks, ids = self._unpack(r, (H, W))
            if ids is None: continue
            for i in np.argsort(scores):            # most confident painted last
                out[t][masks[i]] = remap.setdefault(int(ids[i]), len(remap) + 1)
        return out

def _same_box(a, b, iou_thr):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    area_a, area_b = (a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1])
    return inter / max(area_a + area_b - inter, 1e-6) >= iou_thr or inter / max(min(area_a, area_b), 1e-6) >= 0.7

class DetectThenSAM2(Model):
    """The automatic SAM 2 pipeline: a detector finds objects on every `every`-th frame, SAM 2.1 turns each
    new box into a mask and follows it. SAM 2 cannot find grapes by itself; it only follows what it is shown."""
    tracks = True
    def __init__(self, detector, size=None, ckpt=None, every=4, match_iou=0.4, max_objects=200, backward=False):
        self.detector, self.size, self.ckpt = detector, size or SAM2_SIZE, ckpt
        self.every, self.match_iou, self.max_objects, self.backward = every, match_iou, max_objects, backward
        self.last_prompts = []
    def prepare(self):
        self.detector.prepare(); load_sam2(self.size, self.ckpt)
    def predict(self, clip, prompts=None):
        sess = Sam2Session(load_sam2(self.size, self.ckpt), clip)
        ids = np.zeros((sess.T, sess.H, sess.W), np.uint16); best = np.zeros(ids.shape, np.float16)
        made = []
        for t in range(0, sess.T, self.every):
            boxes, scores, _ = self.detector.detect(clip.image(t))
            known = [(s[1].start, s[0].start, s[1].stop, s[0].stop) for s in ndimage.find_objects(ids[t]) if s is not None]
            new = []
            for b in boxes[np.argsort(-scores)]:
                if len(made) + len(new) >= self.max_objects: break
                if any(_same_box(b, k, self.match_iou) for k in known): continue
                known.append(tuple(b))
                new.append(dict(obj=len(made) + len(new) + 1, frame=t, box=[float(v) for v in b]))
            if new:
                sess.add(new, ids, best, backward=self.backward); made += new
        self.last_prompts = made
        return ids

class FromFolder(Model):
    """Score predictions written by any other program: <folder>/<clip>/<frame>.png, pixel value = object id."""
    def __init__(self, folder, tracks=False):
        self.folder, self.tracks = Path(folder), tracks
    def predict(self, clip, prompts=None):
        lab = clip.labels(root=self.folder)
        if lab is None:
            raise FileNotFoundError(f"{self.folder / clip.name} needs one PNG per frame, named like the frames")
        return lab

class SAM3Text(Model):
    """SAM 3.1 (or SAM 3) with a text prompt, through Meta's own `sam3` package. Needs that package installed
    and Hugging Face access to the gated weights. NOT run while this pipeline was tested. See Appendix B."""
    tracks = True
    def __init__(self, text=None, version="sam3.1", max_num_objects=64, **builder_args):
        self.text, self._p = text or OBJECT_NAME, None
        self.args = dict(version=version, **builder_args)
        if version == "sam3.1":
            self.args["max_num_objects"] = max_num_objects      # Meta's default of 16 is too few for a vine row
    def prepare(self):
        if self._p is None:
            import sam3.model_builder as mb
            if hasattr(mb, "build_sam3_predictor"):              # March 2026 and later: SAM 3 and SAM 3.1
                self._p = mb.build_sam3_predictor(**self.args)
            else:                                                # original SAM 3 release
                self._p = mb.build_sam3_video_predictor(**{k: v for k, v in self.args.items()
                                                            if k not in ("version", "max_num_objects")})
    def predict(self, clip, prompts=None):
        self.prepare()
        T = len(clip); H, W = clip.hw
        out = np.zeros((T, H, W), np.uint16)
        sid = self._p.handle_request(dict(type="start_session", resource_path=str(clip.dir)))["session_id"]
        try:
            self._p.handle_request(dict(type="add_prompt", session_id=sid, frame_index=0, text=self.text))
            for resp in self._p.handle_stream_request(dict(type="propagate_in_video", session_id=sid)):
                o = resp["outputs"]
                for k, m in zip(np.asarray(o["out_obj_ids"]).tolist(), np.asarray(o["out_binary_masks"])):
                    m = np.asarray(m, bool)
                    if m.shape != (H, W):
                        m = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST) > 0
                    out[resp["frame_index"]][m] = int(k) + 1
        finally:
            self._p.handle_request(dict(type="close_session", session_id=sid))
        return out

# ---- describing models in plain data (used by the HPCC jobs and the lineup files) ------------------------
def _project_path(p):
    """A path written in a lineup file: absolute, ~/…, or relative to the project folder."""
    p = Path(os.path.expandvars(str(p))).expanduser()
    return p if p.is_absolute() else ROOT / p

def build_model(spec):
    """Make a model from a plain dict, e.g. one entry of a lineup YAML file:
      {type: sam2, size: base_plus, checkpoint: models/x.pt, prompt: box}      SAM 2.1, pretrained or fine-tuned
      {type: ultralytics, weights: yoloe-26m-seg.pt, text: grape cluster}      YOLO / YOLOE (automatic)
      {type: detect_then_sam2, detector: {...}, checkpoint: models/x.pt}       a detector finds, SAM 2.1 follows
      {type: folder, path: predictions/sam3_text, tracks: true}                masks written by any other program
      {type: sam3, text: grape cluster}                                        SAM 3.1 (untested, gated weights)
      {type: python, class: my_models.py:MyModel, args: {...}}                 your own Model subclass"""
    spec = dict(spec); kind = spec.pop("type").lower()
    ckpt = spec.pop("checkpoint", None)
    ckpt = _project_path(ckpt) if ckpt else None
    if kind == "sam2":
        return SAM2Prompted(spec.get("size"), ckpt, spec.get("prompt", "box"))
    if kind in ("ultralytics", "yolo", "yoloe"):
        w = str(spec.pop("weights"))
        w = str(_project_path(w)) if ("/" in w or _project_path(w).exists()) else w      # bare names: Ultralytics fetches them
        return UltralyticsModel(w, **spec)
    if kind == "detect_then_sam2":
        return DetectThenSAM2(build_model(spec.pop("detector")), spec.pop("size", None), ckpt, **spec)
    if kind == "folder":
        return FromFolder(_project_path(spec["path"]), tracks=bool(spec.get("tracks", False)))
    if kind == "sam3":
        return SAM3Text(**spec)
    if kind == "python":
        target, args = spec.pop("class"), spec.pop("args", {}) or {}
        mod_name, cls_name = target.rsplit(":", 1)
        import importlib, importlib.util
        if mod_name.endswith(".py"):
            path = _project_path(mod_name) if not Path(mod_name).expanduser().exists() else Path(mod_name).expanduser()
            spec_ = importlib.util.spec_from_file_location(path.stem, path)
            mod = importlib.util.module_from_spec(spec_)
            sys.modules[path.stem] = mod                       # its classes can then subclass Model from here
            mod.__dict__.update({k: v for k, v in globals().items() if k in ("Model", "np", "cv2", "Clip")})
            spec_.loader.exec_module(mod)
        else:
            mod = importlib.import_module(mod_name)
        return getattr(mod, cls_name)(**args)
    raise ValueError(f"unknown model type '{kind}' (sam2, ultralytics, detect_then_sam2, folder, sam3, python)")

def load_lineup(path):
    """Read a lineup YAML file → {name: Model}. See configs/lineup.yaml for a commented example.
    A `python` model's file may be given relative to the lineup file itself."""
    import yaml
    path = Path(path).expanduser()
    entries = yaml.safe_load(path.read_text())["models"] or []
    out = {}
    for e in entries:
        e = dict(e)
        if e.pop("skip", False):
            continue
        if e.get("type") == "python" and ".py:" in e.get("class", ""):
            f, cls = e["class"].rsplit(":", 1)
            if not Path(f).expanduser().is_absolute() and (path.parent / f).exists():
                e["class"] = f"{(path.parent / f).resolve()}:{cls}"
        name = e.pop("name")                 # (popped before build_model sees the entry)
        out[name] = build_model(e)
    return out

# ---- run + score ------------------------------------------------------------------------------------
PROMPT_VISIBLE_FRAC = 0.5    # prompt an object on the first frame where half of its largest view is in frame
MIN_OBJECT_AREA = 100        # automatic track: ignore specks smaller than this many pixels (labels and predictions)

def run_model(name, model, clips):
    """Predict every clip and save the result under predictions/<name>/. Returns seconds per frame."""
    root, secs, n = PRED / _safe(name), 0.0, 0
    model.prepare()
    for clip in tqdm(clips, desc=name, leave=False):
        prompts = prompts_from_labels(clip.labels(), model.prompt, PROMPT_VISIBLE_FRAC) if model.needs_prompts else None
        t0 = time.time()
        pred = np.asarray(model.predict(clip, prompts))
        secs += time.time() - t0; n += len(clip)
        assert pred.shape == (len(clip), *clip.hw), f"{name}: expected {(len(clip), *clip.hw)}, got {pred.shape}"
        clip.save_labels(pred, root=root)
    return secs / max(n, 1)

def score_model(name, model, clips):
    """Compare saved predictions with the labels. Returns (summary row, detail table)."""
    root, obj_rows, frame_rows, uniq = PRED / _safe(name), [], [], [0, 0]
    for clip in clips:
        gt, pred = clip.labels(), clip.labels(root=root)
        if model.needs_prompts:
            prompts = prompts_from_labels(gt, "box", PROMPT_VISIBLE_FRAC)        # same frames for every prompt type
            obj_rows += [dict(model=name, clip=clip.name, **r) for r in score_prompted(gt, pred, prompts)]
        else:
            frame_rows += [dict(model=name, clip=clip.name, **r) for r in score_automatic(gt, pred, 0.5, MIN_OBJECT_AREA)]
            for i, arr in enumerate((gt, pred)):
                k, c = np.unique(arr, return_counts=True)
                uniq[i] += int(((k != 0) & (c >= MIN_OBJECT_AREA)).sum())
    if not obj_rows and not frame_rows:
        raise RuntimeError("nothing to score: these clips have no labelled objects")
    if model.needs_prompts:
        d = pd.DataFrame(obj_rows)
        J, F = d["J"].mean(), d["F"].mean()
        row = {"model": name, "kind": "prompted", "prompt": model.prompt, "objects": len(d),
               "J&F": (J + F) / 2, "J": J, "F": F, "J&F (DAVIS)": (d["J_all"].mean() + d["F_all"].mean()) / 2,
               "IoU on prompt frame": np.nan if model.prompt == "mask" else d["iou_prompt_frame"].mean()}
    else:
        d = pd.DataFrame(frame_rows)
        tp, fp, fn = d["tp"].sum(), d["fp"].sum(), d["fn"].sum()
        diff = d["n_pred"] - d["n_gt"]
        P, R = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        row = {"model": name, "kind": "automatic", "precision": 100 * P, "recall": 100 * R,
               "F1": 100 * 2 * P * R / max(P + R, 1e-9), "mask IoU (matched)": 100 * d["iou_sum"].sum() / max(tp, 1),
               "count error / frame": diff.abs().mean(), "count bias / frame": diff.mean(),
               "objects / frame (truth)": d["n_gt"].mean(),
               "tracks found / true": f"{uniq[1]} / {uniq[0]}" if model.tracks else "n/a"}
    return row, d

def _data_signature(clips):
    """Short fingerprint of the clips and their labels: rows scored on different data are never shown together."""
    import hashlib
    h = hashlib.md5()
    for c in sorted(clips, key=lambda c: c.name):
        h.update(c.name.encode())
        for f in sorted((ANN / c.name).glob("*.png")): h.update(f.read_bytes())
    return h.hexdigest()[:8]

def benchmark(models, split="val", clips=None):
    """Run and score every model in `models` (a dict name → Model) on the labelled clips of `split`.
    Rows add up across calls, so the pretrained models scored earlier sit next to the fine-tuned ones."""
    clips = clips or get_clips(split, labelled=True)
    if not clips:
        raise RuntimeError(f"No labelled '{split}' clips yet. Label some in Step 3 and mark them done "
                           "(Step 3c shows what is missing).")
    sig, rows, details = _data_signature(clips), [], {}
    for name, model in models.items():
        try:
            spf = run_model(name, model, clips)
            row, details[name] = score_model(name, model, clips)
            rows.append({**row, "sec / frame": spf, "data": sig})
        except Exception as e:                       # one broken model should not sink the whole table
            print(f"✗ {name}: {type(e).__name__}: {str(e)[:300]}")
    RESULTS.mkdir(exist_ok=True)
    f, fd = RESULTS / f"leaderboard_{split}.csv", RESULTS / f"leaderboard_{split}_details.csv"
    old = pd.read_csv(f) if f.exists() and f.stat().st_size > 5 else pd.DataFrame()
    if len(old):
        stale = int((old["data"] != sig).sum())
        if stale: print(f"({stale} earlier row(s) were scored on a different set of clips or labels and were dropped)")
        old = old[(old["data"] == sig) & ~old["model"].isin([r["model"] for r in rows])]
    board = pd.concat([old, pd.DataFrame(rows)], ignore_index=True) if len(old) else pd.DataFrame(rows)
    if len(board): board.to_csv(f, index=False)
    if details:
        new = pd.concat(details.values()).assign(data=sig)
        if fd.exists():
            od = pd.read_csv(fd); new = pd.concat([od[(od["data"] == sig) & ~od["model"].isin(list(details))], new])
        new.to_csv(fd, index=False)
    print(f"scored on {len(clips)} {split} clip(s), {sum(len(c) for c in clips)} frames · table saved to {f}")
    show_board(board)
    own = [c.name for c in clips if _load_prompts(c.name)]
    if own and len(board) and (board["kind"] == "prompted").any():
        print(f"\n⚠️  {len(own)} of these {len(clips)} clips were labelled with SAM 2.1's help (labelling screen or box imports), so SAM 2.1 is partly being "
              "graded against its own drawing. Its 'box' rows read high for that reason alone. Trust differences on the "
              "click rows and in the automatic table more, and see 'Keeping the score honest' under Step 3.")
    return board

def clear_leaderboard(split="val"):
    """Start the table over (the saved predictions stay)."""
    for f in RESULTS.glob(f"leaderboard_{split}*.csv"): f.unlink()

def show_board(board):
    for kind, cols, key in (("prompted", ["model", "prompt", "J&F", "J", "F", "J&F (DAVIS)", "IoU on prompt frame", "objects", "sec / frame"], "J&F"),
                            ("automatic", ["model", "F1", "precision", "recall", "mask IoU (matched)", "count error / frame",
                                           "count bias / frame", "objects / frame (truth)", "tracks found / true", "sec / frame"], "F1")):
        part = board[board["kind"] == kind] if len(board) else board
        if len(part):
            print(f"\n{kind.upper()} models" + (" — shown each object once, must follow it" if kind == "prompted"
                                                   else " — must find every object themselves"))
            display(part[cols].sort_values(key, ascending=False).round(2).reset_index(drop=True))

In [ ]:
#@title Tools: fine-tuning SAM 2.1 (Meta's trainer) and training a YOLO detector  { display-mode: "form" }
_LADDER = [(8, 3), (6, 3), (4, 3), (4, 2), (4, 1), (2, 1)]      # (frames per sample, objects per sample)

def _auto_train_plan(encoder="auto"):
    if encoder == "yes":  freeze = False
    elif encoder == "no": freeze = True
    else:                 freeze = GPU_GB < 38
    start = 0 if GPU_GB >= 20 else 3 if DEVICE == "cuda" else 5
    return freeze, start

def write_train_config(run, train_clips, size, steps, lr, freeze, frames, objects, init_ckpt=None,
                       steps_per_epoch=100, data_root=None, workers=2 if DEVICE == "cuda" else 0, amp_dtype="bfloat16"):
    """Start from Meta's official fine-tuning recipe and change only what a small single-GPU run needs."""
    from omegaconf import OmegaConf
    cfg_dir = SAM2_REPO / "sam2" / "configs" / "sam2.1_training"
    cfg = OmegaConf.load(cfg_dir / "sam2.1_hiera_b+_MOSE_finetune.yaml")
    inf = OmegaConf.load(SAM2_REPO / "sam2" / SAM2_MODELS[size]["cfg"])
    run_dir = RUNS / run; run_dir.mkdir(parents=True, exist_ok=True)
    data_root = Path(data_root or DATA)

    # backbone of the chosen size (the recipe file is written for base_plus)
    enc = cfg.trainer.model.image_encoder
    enc.trunk = OmegaConf.merge(inf.model.image_encoder.trunk, {"drop_path_rate": 0.0 if freeze else 0.1})
    enc.neck.backbone_channel_list = inf.model.image_encoder.neck.backbone_channel_list
    cfg.trainer.model.freeze_image_encoder = bool(freeze)
    # The recipe may hand the model ground-truth masks on up to 2 frames; with only 2 frames per sample that
    # would leave nothing to predict (and nothing to learn from), so keep at least one frame to track.
    cfg.trainer.model.num_init_cond_frames_for_train = max(1, min(2, frames - 1))

    # data: clips with enough frames train as video, single frames (imported images) train as images
    video = [c for c in train_clips if len(c) >= max(frames, 2)]
    image = [c for c in train_clips if len(c) == 1]
    skipped = [c.name for c in train_clips if c not in video and c not in image]
    template = cfg.trainer.data.train.datasets[0]
    entries, n_samples = [], 0
    for kind, clips, n_frames in (("video", video, frames), ("image", image, 1)):
        if not clips: continue
        (run_dir / f"train_{kind}.txt").write_text("\n".join(c.name for c in clips) + "\n")
        mult = max(1, round(min(steps_per_epoch, steps) / (len(video) + len(image))))   # ≈ steps_per_epoch samples per epoch
        e = OmegaConf.create(OmegaConf.to_container(template, resolve=False))
        d = e.dataset.datasets[0]
        d.video_dataset.img_folder = str(data_root / "JPEGImages")
        d.video_dataset.gt_folder = str(data_root / "Annotations")
        d.video_dataset.file_list_txt = str(run_dir / f"train_{kind}.txt")
        d.sampler.num_frames, d.sampler.max_num_objects, d.multiplier = n_frames, objects, mult
        entries.append(e); n_samples += len(clips) * mult
    assert entries, "no usable training clips"
    cfg.trainer.data.train.datasets = entries
    cfg.trainer.data.train.batch_sizes = [1] * len(entries)
    if len(entries) > 1:
        cfg.trainer.data.train.dataset_prob = [0.5, 0.5]
    cfg.trainer.data.train.num_workers = workers

    epochs = max(1, math.ceil(steps / n_samples))
    cfg.scratch.update(dict(resolution=int(SAM2_IMAGE_SIZE), num_frames=frames, max_num_objects=objects,
                            num_epochs=epochs, num_train_workers=workers, base_lr=float(lr),
                            vision_lr=float(lr) * 0.6, train_batch_size=1))
    cfg.dataset.update(dict(img_folder=str(data_root / "JPEGImages"), gt_folder=str(data_root / "Annotations"),
                            file_list_txt=str(run_dir / "train_video.txt")))
    cfg.trainer.accelerator = DEVICE
    cfg.trainer.distributed.backend = "nccl" if DEVICE == "cuda" else "gloo"
    cfg.trainer.optim.amp.enabled = DEVICE == "cuda"
    cfg.trainer.optim.amp.amp_dtype = amp_dtype      # bfloat16 needs an A100/L4/L40S/H100-class GPU; T4 and V100 use float16
    cfg.trainer.logging.log_freq = int(max(1, min(10, steps // 50)))   # short runs log every step, for the loss plot
    cfg.trainer.checkpoint.model_weight_initializer.state_dict.checkpoint_path = str(init_ckpt or sam2_checkpoint(size))
    cfg.launcher.gpus_per_node = 1
    cfg.launcher.experiment_log_dir = str(run_dir)

    name = f"fruit_{_safe(run)}.yaml"
    (cfg_dir / name).write_text("# @package _global_\n\n" + OmegaConf.to_yaml(cfg))
    plan = dict(run=run, size=size, image_size=int(SAM2_IMAGE_SIZE), steps=epochs * n_samples, epochs=epochs,
                steps_per_epoch=n_samples, lr=float(lr), freeze_encoder=bool(freeze), frames=frames, objects=objects,
                video_clips=len(video), image_clips=len(image), skipped=skipped,
                amp_dtype=amp_dtype if DEVICE == "cuda" else "off",
                train_clips=[c.name for c in video + image], sam2_commit=SAM2_COMMIT)
    return f"configs/sam2.1_training/{name}", plan

def _launch(config_name, run_dir, total_steps):
    """Run Meta's trainer as a subprocess; print one short progress line per log tick. Returns (ok, log tail)."""
    env = dict(os.environ, PYTHONPATH=str(SAM2_REPO) + os.pathsep + os.environ.get("PYTHONPATH", ""))
    args = ["training/train.py", "-c", config_name, "--use-cluster", "0", "--num-gpus", "1"]
    cmd = [sys.executable] + args
    if DEVICE != "cuda":       # Meta's trainer resets GPU memory statistics every step; make that a no-op on CPU
        cmd = [sys.executable, "-c", "import sys, runpy, torch; torch.cuda.reset_peak_memory_stats = lambda *a, **k: None; "
               f"sys.argv = {args!r}; runpy.run_path('training/train.py', run_name='__main__')"]
    log = open(run_dir / "console.log", "a")
    proc = subprocess.Popen(cmd, cwd=SAM2_REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail, pat = [], re.compile(r"Train Epoch: \[(\d+)\]\[\s*(\d+)/\s*(\d+)\] \| Batch Time: [\d.]+ \(([\d.]+)\).*?"
                               r"Mem \(GB\): [\d.]+ \([\d.]+/([\d.]+)\).*?Losses/train_all_loss: [\d.eE+-]+ \(([\d.eE+-]+)\)")
    shown, every = -10**9, max(10, total_steps // 15)          # about 15 progress lines per run
    for line in proc.stdout:
        log.write(line); tail = (tail + [line])[-60:]
        m = pat.search(line)
        if m:
            ep, it, per = int(m[1]), int(m[2]), int(m[3])
            log.flush()
            step = ep * per + it
            if step - shown < every and step < total_steps - 1:
                continue
            shown = step
            print(f"  step {step:>5}/{total_steps} · average loss this epoch {float(m[6]):.3f} · {float(m[4]):.1f} s/step"
                  + (f" · peak GPU memory {float(m[5]):.0f} GB" if DEVICE == "cuda" else ""), flush=True)
    proc.wait(); log.close()
    return proc.returncode == 0, "".join(tail)

def finetune(run="grapes_v1", size=None, steps=1500, lr=5e-6, encoder="auto", clips=None, init_ckpt=None,
             amp_dtype="auto"):
    """Fine-tune SAM 2.1 with Meta's trainer on the labelled train clips and return the path of the exported
    weights. Re-running with the same name resumes an interrupted run, or reuses the finished weights.
    encoder: "auto" | "no" (freeze the image backbone, fits 16 GB) | "yes" (train it too, ~40 GB+).
    amp_dtype: "auto" (bfloat16 on A100/L4/L40S/H100-class GPUs, float16 on T4/V100) | "bfloat16" | "float16"."""
    size = size or SAM2_SIZE
    clips = clips or get_clips("train", labelled=True)
    if not clips:
        raise RuntimeError("No labelled training clips yet. Label some in Step 3 and mark them done.")
    out = WEIGHTS / f"sam2.1_{size}_{_safe(run)}.pt"
    if out.exists():
        print(f"{out.name} already exists, reusing it. Pick a new run name to train again."); return out
    over = [c.name for c in clips if int(c.labels().max()) > 255]
    if over:
        raise RuntimeError(f"Meta's trainer reads at most 255 objects per clip; these have more: {over[:5]}")
    run_dir = RUNS / run
    data_root = DATA
    if "/drive/" in str(DATA):            # Drive is slow for thousands of small reads: train from a fresh local copy
        data_root = BASE / "train_data_local"
        shutil.rmtree(data_root, ignore_errors=True)
        for sub in ("JPEGImages", "Annotations"):
            for c in clips:
                shutil.copytree(DATA / sub / c.name, data_root / sub / c.name)
    freeze, rung = _auto_train_plan(encoder)
    if amp_dtype == "auto":
        amp_dtype = "bfloat16" if BF16_OK else "float16"
    shortest = min((len(c) for c in clips if len(c) > 1), default=1)
    while rung < len(_LADDER) - 1 and _LADDER[rung][0] > shortest: rung += 1
    while True:
        frames, objects = _LADDER[rung]
        config_name, plan = write_train_config(run, clips, size, steps, lr, freeze, min(frames, max(shortest, 1)),
                                               objects, init_ckpt, data_root=data_root, amp_dtype=amp_dtype)
        old = json.loads((run_dir / "plan.json").read_text()) if (run_dir / "plan.json").exists() else plan
        if old != plan:                    # same name, different settings: do not resume the old attempt
            print("  settings changed since the last attempt under this name, starting it over")
            for sub in ("checkpoints", "logs", "tensorboard"): shutil.rmtree(run_dir / sub, ignore_errors=True)
        (run_dir / "plan.json").write_text(json.dumps(plan, indent=1))
        print(f"training {size}: {plan['steps']} steps on {plan['video_clips']} video + {plan['image_clips']} image clip(s), "
              f"{plan['frames']} frames × {objects} objects per step, encoder {'frozen' if freeze else 'trained'}, lr {lr:g}"
              + (f", {amp_dtype} on {GPU_NAME}" if DEVICE == "cuda" else ", CPU"))
        if plan["skipped"]: print("  skipped (too short for video training):", plan["skipped"])
        t0 = time.time()
        ok, tail = _launch(config_name, run_dir, plan["steps"])
        if ok: break
        if "out of memory" in tail.lower() and rung < len(_LADDER) - 1:
            rung += 1
            shutil.rmtree(run_dir / "checkpoints", ignore_errors=True)
            print(f"  GPU ran out of memory; retrying with {_LADDER[rung][0]} frames × {_LADDER[rung][1]} objects")
            continue
        if "attempting to stop training" in tail and amp_dtype == "float16" and DEVICE == "cuda":
            amp_dtype = "bfloat16"            # float16 overflowed: slower but steadier on this GPU
            shutil.rmtree(run_dir / "checkpoints", ignore_errors=True)
            print("  the loss became infinite in float16; retrying in bfloat16 (slower on this GPU)")
            continue
        print(tail[-3000:]); raise RuntimeError(f"training failed, full log: {run_dir / 'console.log'}")
    sd = torch.load(run_dir / "checkpoints" / "checkpoint.pt", map_location="cpu", weights_only=True)["model"]
    # weights only, so it loads like any SAM 2.1 checkpoint; size and resolution travel with the file
    torch.save({"model": sd, "size": size, "image_size": int(SAM2_IMAGE_SIZE)}, out)
    plan["minutes"] = round((time.time() - t0) / 60, 1)
    out.with_suffix(".json").write_text(json.dumps(plan, indent=1))  # the full recipe behind these weights
    print(f"done in {plan['minutes']} min → {out}")
    return out

def plot_training(run="grapes_v1"):
    """Training loss over the run, read from the trainer's log (one reading every 10 steps). Single readings
    jump around because each step sees one clip; the line is their running mean. Full detail: TensorBoard logs
    in runs/<name>/tensorboard."""
    f = RUNS / run / "console.log"
    if not f.exists():
        print("no training log for this run in", f.parent); return
    pts = re.findall(r"Train Epoch: \[(\d+)\]\[\s*(\d+)/\s*(\d+)\].*?Losses/train_all_loss: ([\d.eE+-]+)", f.read_text())
    last = {}
    for ep, it, per, loss in pts: last[int(ep) * int(per) + int(it)] = float(loss)     # a resumed run logs steps twice
    if len(last) < 3:
        print(f"only {len(last)} loss reading(s): too short a run to draw a curve"); return
    x = np.array(sorted(last)); y = np.array([last[k] for k in x])
    k = max(1, min(10, len(y) // 3)); smooth = np.convolve(np.pad(y, (k - 1, 0), mode="edge"), np.ones(k) / k, mode="valid")
    plt.figure(figsize=(7, 2.8)); plt.plot(x, y, ".", alpha=0.35, label="one step"); plt.plot(x, smooth, label=f"mean of last {k} readings")
    plt.xlabel("training step"); plt.ylabel("loss"); plt.legend(); plt.tight_layout()
    if _in_notebook():
        plt.show()
    else:                                  # command line / batch job: keep the picture next to the run
        plt.savefig(f.parent / "loss.png", dpi=120); plt.close(); print("loss curve:", f.parent / "loss.png")

def train_detector(base="yolo26n-seg.pt", epochs=100, imgsz=1280, name="yolo_detector", every=2):
    """Train a YOLO segmentation model on the labelled clips (same train / val split). Returns the path of its
    best weights. `every`: use every n-th frame of each clip (neighbouring frames are near-copies)."""
    ul = _ultralytics()
    best = RUNS / name / "weights" / "best.pt"
    if best.exists():
        print(f"{best} already exists, reusing it. Pick a new name to train again."); return best
    data = export_yolo_seg(every=every)
    model = ul.YOLO(base)
    model.train(data=str(data), epochs=int(epochs), imgsz=int(imgsz), project=str(RUNS),
                name=name, exist_ok=True, device=0 if DEVICE == "cuda" else "cpu", workers=2, plots=False)
    return Path(model.trainer.best)

In [ ]:
#@title Tools: side-by-side comparisons and running a pipeline over long video  { display-mode: "form" }
def compare(clip, names, frames=4, video=True):
    """Side by side: the labels, then each named model's saved predictions, on the same frames."""
    clip = Clip(clip) if isinstance(clip, str) else clip
    rows = [("labels (ground truth)", clip.labels())] + [(n, clip.labels(root=PRED / _safe(n))) for n in names]
    rows = [(n, a) for n, a in rows if a is not None]
    picks = np.linspace(0, len(clip) - 1, min(frames, len(clip))).round().astype(int)
    h, w = clip.hw
    fig, ax = plt.subplots(len(rows), len(picks), figsize=(4.2 * len(picks), 4.2 * h / w * len(rows) + 0.3), squeeze=False)
    for r, (n, a) in enumerate(rows):
        for c, t in enumerate(picks):
            ax[r, c].imshow(overlay(clip.image(t), a[t], ids=False)); ax[r, c].axis("off")
            ax[r, c].set_title(f"{n} · frame {t}" if c == 0 else f"frame {t}", fontsize=8, loc="left")
    plt.tight_layout()
    if _in_notebook():
        plt.show()
    else:
        plt.savefig(RESULTS / f"compare_{clip.name}.png", dpi=110); plt.close()
    if video and len(clip) > 1:
        def tiles():
            for t in range(len(clip)):
                row = []
                for n, a in rows:
                    im = overlay(clip.image(t), a[t], ids=False)
                    im = cv2.resize(im, (480, round(480 * h / w)), interpolation=cv2.INTER_AREA)
                    cv2.putText(im, n[:46], (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3, cv2.LINE_AA)
                    cv2.putText(im, n[:46], (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
                    row.append(im)
                yield np.concatenate(row, axis=1)
        out = write_video(tiles(), RESULTS / f"compare_{clip.name}.mp4", fps=4)
        if _in_notebook():
            display(Video(str(out), embed=True, html_attributes="controls loop"))
        else:
            print("comparison video:", out)

def run_on_video(model, video, start_s=0, seconds=10, fps=6, max_side=1280, chunk=24, min_track_frames=2):
    """Run an automatic model over part of a video in overlapping chunks and stitch the object ids together.
    Returns (overlay video path, per-frame counts, per-object tracks)."""
    video = Path(video); info = probe_video(video)
    assert info and info["fps"], f"cannot read {video}"
    stride = max(1, round(info["fps"] / fps))
    work = BASE / "video_runs" / f"{_safe(video.stem)}_{int(start_s)}s_{int(seconds)}s"
    shutil.rmtree(work, ignore_errors=True); (work / "all").mkdir(parents=True)
    cap = cv2.VideoCapture(str(video)); cap.set(cv2.CAP_PROP_POS_MSEC, start_s * 1000)
    n, i, limit = 0, 0, max(1, int(seconds * info["fps"] / stride))
    while n < limit and cap.grab():
        if i % stride == 0:
            ok, bgr = cap.retrieve()
            if not ok: break
            sc = max_side / max(bgr.shape[:2])
            if sc < 1: bgr = cv2.resize(bgr, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(work / "all" / f"{n:05d}.jpg"), bgr, [cv2.IMWRITE_JPEG_QUALITY, 95]); n += 1
        i += 1
    cap.release()
    assert n > 0, "no frames could be read from that part of the video"
    model.prepare()
    tracks, per_frame, state = {}, [], dict(last=None, next_id=1)

    def overlays():
        start = 0
        while True:
            stop = min(start + chunk, n)
            cdir = work / f"chunk_{start:05d}"; cdir.mkdir()
            for j, t in enumerate(range(start, stop)):
                try:
                    os.symlink(work / "all" / f"{t:05d}.jpg", cdir / f"{j:05d}.jpg")
                except OSError:                       # Windows without Developer Mode: copy instead
                    shutil.copy2(work / "all" / f"{t:05d}.jpg", cdir / f"{j:05d}.jpg")
            clip = Clip(cdir.name, root=work)
            ids = np.asarray(model.predict(clip)).astype(np.int64)
            lut = np.zeros(int(ids.max()) + 1, np.int64)
            if state["last"] is not None:                     # same frame seen by both chunks: carry ids over
                old_ids, new_ids, iou, _, _ = iou_matrix(state["last"], ids[0])
                if iou.size:
                    r, c = linear_sum_assignment(-iou)
                    for a, b in zip(r, c):
                        if iou[a, b] >= 0.5: lut[new_ids[b]] = old_ids[a]
            for k in np.unique(ids):
                if k and lut[k] == 0:
                    lut[k] = state["next_id"]; state["next_id"] += 1
            ids = lut[ids]
            for j in range(0 if state["last"] is None else 1, len(ids)):
                t = start + j
                k, cnt = np.unique(ids[j], return_counts=True); k = k[(k != 0) & (cnt >= MIN_OBJECT_AREA)]
                per_frame.append(dict(frame=t, second=round(start_s + t * stride / info["fps"], 2), objects_in_view=len(k)))
                for o in k:
                    tr = tracks.setdefault(int(o), dict(object=int(o), first_frame=t, last_frame=t, frames=0))
                    tr["last_frame"] = t; tr["frames"] += 1
                im = overlay(clip.image(j), ids[j])
                yield cv2.resize(im, (960, round(960 * im.shape[0] / im.shape[1])), interpolation=cv2.INTER_AREA)
            state["last"] = ids[-1]
            if stop >= n: break
            start = stop - 1

    out = write_video(overlays(), RESULTS / f"overlay_{_safe(video.stem)}_{int(start_s)}s.mp4", fps=info["fps"] / stride)
    counts = pd.DataFrame(per_frame)
    tr = pd.DataFrame(list(tracks.values()), columns=["object", "first_frame", "last_frame", "frames"])
    tr = tr[tr["frames"] >= min_track_frames].reset_index(drop=True)
    counts.to_csv(out.with_suffix(".counts.csv"), index=False); tr.to_csv(out.with_suffix(".tracks.csv"), index=False)
    print(f"{n} frames · {counts['objects_in_view'].mean():.1f} objects in view on average · "
          f"{len(tr)} distinct objects seen in at least {min_track_frames} frames\noverlay: {out}")
    return out, counts, tr

## Step 1 · Add your videos
Any phone or GoPro format works (`.mp4`, `.mov`, …). Videos are never modified; the notebook only reads frames from them.

In [ ]:
#@title Step 1 · Add your videos  { display-mode: "form" }
#@markdown Three ways to get videos in. Use whichever is easiest, then re-run this cell to see the list.
#@markdown 1. **Upload button** (this cell, Colab only). Fine for clips up to a few hundred MB.
#@markdown 2. **Drag and drop** into the `videos` folder shown below using Colab's Files panel (left sidebar). Best for big GoPro files.
#@markdown 3. **Copy them into Google Drive** at *My Drive / PROJECT_NAME / videos* from any computer.
UPLOAD_NOW = False  #@param {type:"boolean"}

if UPLOAD_NOW and not DEMO:
    if IN_COLAB:
        from google.colab import files
        _cwd = os.getcwd(); os.chdir(VIDEOS)            # the upload widget saves into the current folder
        try:
            files.upload()
        finally:
            os.chdir(_cwd)
    else:
        print(f"Not in Colab: copy your videos into {VIDEOS}")

print(f"videos folder: {VIDEOS}")
_v = list_videos()
if len(_v):
    display(_v)
elif not DEMO:
    print("No videos yet. Add some (see the three options above) and run this cell again.")

## Step 2 · Cut clips, split train / val
Labelling a whole video is not realistic, so the notebook samples short, evenly spaced, non-overlapping **clips** from each video. Defaults: 6 clips per video, 16 frames each at 6 frames per second (about 2.7 seconds), frames shrunk to 1280 px on the long side.

Each clip is tagged **train** or **val** once and keeps that tag forever:
- with 3 or more videos, whole videos are held out for val (the fair way);
- with 1 or 2 videos, the last clips in time of each video are held out (weaker: nearby vines can look alike).

Re-running this step after adding videos only adds clips. It never moves a val clip into train.

In [ ]:
#@title Step 2 · Cut the videos into short clips and split them train / val  { display-mode: "form" }
#@markdown A *clip* is a short run of frames (default 16 frames at 6 fps ≈ 2.7 s). You label clips,
#@markdown SAM 2 trains on clips, and every model is scored on clips it has never seen.
CLIPS_PER_VIDEO = 6      #@param {type:"slider", min:1, max:40, step:1}
CLIP_FRAMES     = 16     #@param {type:"slider", min:8, max:48, step:1}
CLIP_FPS        = 6      #@param {type:"number"}
MAX_SIDE        = 1280   #@param [960, 1280, 1920] {type:"raw"}
VAL_FRACTION    = 0.25   #@param {type:"slider", min:0.1, max:0.5, step:0.05}

if not DEMO:
    for _p in sorted(VIDEOS.iterdir()):
        if _p.suffix.lower() in VIDEO_EXTS:
            _made = cut_clips(_p, CLIPS_PER_VIDEO, CLIP_FRAMES, CLIP_FPS, MAX_SIDE)
            print(f"{_p.name}: {len(_made)} clip(s)")
    if len(clips_table()):
        print("split:", split_clips(VAL_FRACTION))
        display(clips_table())

In [ ]:
#@title Demo data: sample grape clips with ready-made labels (only runs when DEMO is on)  { display-mode: "form" }
if DEMO:
    # Photos and hand-drawn cluster masks come from Embrapa WGISD (Santos et al.), CC BY-NC 4.0:
    # https://github.com/thsant/wgisd. Each photo is turned into a short clip by sliding a window
    # across it, so the "video" has exact ground truth to learn from and to grade against.
    make_demo_clips()
    print("split:", split_clips(VAL_FRACTION))
    display(clips_table())
    show_clip(get_clips("val")[0], title="A demo validation clip with its ground-truth labels")

## Step 3 · Label
A label is a mask per cluster per frame, with the same number for the same cluster across the clip. You do not paint masks. You **draw one box per cluster on one frame**, and SAM 2.1 produces the mask and follows the cluster through the other frames. Then you check it and fix what is wrong.

**Decide once what counts as one object, and write it down.** For example: *one physical cluster = one object; two touching clusters are two objects if you can tell them apart; a cluster split by a leaf is still one object.* Every model is graded against that rule, so it has to be the same rule on every clip.

**How many clips?** Enough val clips that one bad clip does not swing the score (aim for 8 or more, from at least 2 videos), and as many train clips as the team has patience for. Step 6 tells you whether more would help.

### Keeping the score honest
The labelling screen uses SAM 2.1 to draw the masks. If a mask is accepted exactly as SAM 2.1 drew it, then asking SAM 2.1 for that mask again later scores near 100 by construction. Two consequences:

- **Fine-tuning can only learn from what you corrected.** Where pretrained SAM 2.1 was already right, there is nothing to learn. Where it merged two clusters, swallowed a leaf or lost a cluster behind a post, and you fixed it, there is. So corrections are the valuable part of labelling, not a chore to skip.
- **SAM 2.1's box-prompt rows are flattered on labels it drew itself.** For comparing models, lean on the click rows and on the automatic table (did it find the clusters, did it count them right). For an independent check of mask quality, hand-trace the clusters on a few val frames in Roboflow or CVAT *without* their SAM assist and import them (Appendix A); those frames are then neutral ground for every model.

Labels made elsewhere (Roboflow, CVAT, Label Studio) can be imported instead of, or in addition to, labelling here: see Appendix A.

In [ ]:
#@title Step 3a · Optional: let a detector draft the labels so you only correct them  { display-mode: "form" }
#@markdown A text-prompted detector (YOLOE) proposes boxes, SAM 2.1 turns them into tracked masks. Drafts stay
#@markdown **unused** until you open each clip below and press *mark done*.
DRAFT_LABELS  = False             #@param {type:"boolean"}
DRAFT_WEIGHTS = "yoloe-26m-seg.pt"  #@param {type:"string"}
DRAFT_TEXT    = "grape cluster"   #@param {type:"string"}
DRAFT_CONF    = 0.2               #@param {type:"slider", min:0.05, max:0.8, step:0.05}

if DRAFT_LABELS and not DEMO:
    draft_labels(DetectThenSAM2(UltralyticsModel(DRAFT_WEIGHTS, text=DRAFT_TEXT, conf=DRAFT_CONF), every=4))

In [ ]:
#@title Step 3b · Label: box each cluster once, SAM 2.1 follows it through the clip  { display-mode: "form" }
#@markdown - **Drag a box** around a cluster on any frame → a new object, tracked forwards and backwards through the clip.
#@markdown - Step through the frames (◀ ▶). A cluster that only comes into view later: go to that frame and box it there.
#@markdown - **Wrong mask on some frame?** Click the object (click mode *Select*), press **Re-box selected**, and drag the
#@markdown   correct box on that frame. This is the reliable fix. *Add to / Remove from selected* clicks are gentle nudges;
#@markdown   SAM often needs a few of them. **Delete selected** removes the object from the whole clip. **Undo** steps back.
#@markdown - When every cluster in every frame is right, press **Clip checked: mark done**, then pick the next clip in the menu.
#@markdown
#@markdown Label **every** cluster you can see in the val clips: a model that finds a real cluster you skipped gets marked wrong.
if DEMO:
    print("Demo mode: the sample clips already have labels. With your own videos the labelling screen opens here.")
else:
    label()

In [ ]:
#@title Step 3c · Check the dataset  { display-mode: "form" }
dataset_report();

## Step 4 · Score the starting line-up
Every model writes its masks to `predictions/<model name>/` in the same format as the labels, and one scoring function grades them all. That is the whole trick behind "evaluate any model".

**Prompted table** (models that are shown each cluster once and must follow it):

| column | meaning |
|---|---|
| **J&F** | Headline. Average of **J** (mask overlap, IoU) and **F** (how well the outlines agree), over the frames *after* the hint where the cluster is in view. 100 is perfect. |
| J&F (DAVIS) | The same, but also counting frames where model and label agree the cluster has left the picture as perfect. This is how papers report it, so it reads higher. |
| IoU on prompt frame | Mask quality on the very frame the model was shown the cluster. This is what matters when a detector supplies the boxes. |

**Automatic table** (models that get only pixels):

| column | meaning |
|---|---|
| **F1**, precision, recall | A predicted cluster counts as correct if it overlaps a labelled one by IoU ≥ 0.5. Precision: of what it drew, how much is real. Recall: of what is real, how much it found. |
| mask IoU (matched) | Mask quality on the clusters it did find. |
| count error / frame | Average of \|predicted − true\| clusters per frame. Bias shows the direction (+ = over-counting). |
| tracks found / true | Distinct cluster ids over whole clips. The number closest to what a yield estimate needs. |

The prompted hints are made from the labels the same way for every model: one hint per cluster, on the first frame where at least half of its largest view is in frame.

In [ ]:
#@title Step 4 · Score the starting line-up on the val clips, before any training  { display-mode: "form" }
#@markdown Add or remove lines in `LINEUP` to compare whatever you like: other SAM 2.1 sizes, your own YOLO
#@markdown weights, predictions written by another notebook (`FromFolder`), SAM 3 (see Appendix B).
#@markdown
#@markdown The *automatic* rows use a YOLOE detector steered by a text prompt. Wording matters: in the team's earlier
#@markdown YOLO test "green grapes" scored higher than "grape cluster" on the sample clip. Try both; the table decides.
INCLUDE_AUTOMATIC = True          #@param {type:"boolean"}
DETECTOR_WEIGHTS  = "yoloe-26m-seg.pt"   #@param {type:"string"}
DETECTOR_TEXT     = "grape cluster"      #@param {type:"string"}
DETECTOR_CONF     = 0.2           #@param {type:"slider", min:0.05, max:0.8, step:0.05}

LINEUP = {
    f"SAM 2.1 {SAM2_SIZE} · pretrained · box":   SAM2Prompted(prompt="box"),      # hint = a box, like a detector gives
    f"SAM 2.1 {SAM2_SIZE} · pretrained · click": SAM2Prompted(prompt="point"),    # hint = one click
    # "SAM 2.1 large · pretrained · box":        SAM2Prompted("large", prompt="box"),
    # "my YOLO-seg":                             UltralyticsModel("/path/to/best.pt"),
    # "from another notebook":                   FromFolder("/path/to/predictions", tracks=True),
}
if INCLUDE_AUTOMATIC:
    DETECTOR = UltralyticsModel(DETECTOR_WEIGHTS, text=DETECTOR_TEXT or None, conf=DETECTOR_CONF)
    LINEUP[f"{Path(DETECTOR_WEIGHTS).stem} '{DETECTOR_TEXT}'"] = DETECTOR
    LINEUP[f"{Path(DETECTOR_WEIGHTS).stem} '{DETECTOR_TEXT}' → SAM 2.1 {SAM2_SIZE} pretrained"] = DetectThenSAM2(DETECTOR)

board = benchmark(LINEUP)

## Step 5 · Fine-tune SAM 2.1
This runs Meta's own training code on the train clips, starting from the pretrained weights. The recipe is Meta's official fine-tuning recipe with four changes for a single small GPU: one GPU instead of eight, the image backbone frozen by default, fewer frames and objects per step when memory is short, and a step budget instead of 40 passes over a large dataset.

- If the GPU runs out of memory the cell retries by itself with fewer frames per step.
- A run is saved under `runs/<RUN_NAME>/` after every ~100 steps, so re-running the cell after a disconnect resumes it.
- The result is `models/sam2.1_<size>_<RUN_NAME>.pt` plus a small `.json` recording exactly what produced it. It loads like any SAM 2.1 checkpoint.
- To try other settings, change `RUN_NAME` so the earlier result is kept for comparison.

**Step 5b (optional)** trains a YOLO detector on the same labels. If the automatic table in Step 4 shows low recall, the detector is what is holding the pipeline back, and this is the fix. Its rows appear in Step 6, and Step 7 uses it automatically.

In [ ]:
#@title Step 5 · Fine-tune SAM 2.1 on your labelled training clips  { display-mode: "form" }
#@markdown Uses Meta's own training code. `TRAIN_STEPS` is the main dial: more steps = more adaptation and more
#@markdown risk of memorising a small dataset, so always compare with the pretrained model on the val clips.
RUN_NAME      = "grapes_v1"   #@param {type:"string"}
TRAIN_STEPS   = 1500          #@param {type:"integer"}
LEARNING_RATE = 5e-6          #@param {type:"number"}
TRAIN_ENCODER = "auto"        #@param ["auto", "no", "yes"]
#@markdown `TRAIN_ENCODER`: "no" freezes the image backbone and trains only the mask decoder and memory
#@markdown (fits a 16 GB T4). "yes" trains everything (needs roughly 40 GB or more). "auto" picks by GPU size.
#@markdown Long runs fit better in an HPCC batch job (`hpcc/train.sb`). Give it the same RUN_NAME and this cell
#@markdown just picks up the finished weights from the project's `models` folder.

if DEMO:
    TRAIN_STEPS = min(TRAIN_STEPS, 150)       # enough to prove the plumbing, not enough to judge fine-tuning
FINETUNED = finetune(RUN_NAME, steps=TRAIN_STEPS, lr=LEARNING_RATE, encoder=TRAIN_ENCODER)
plot_training(RUN_NAME)

In [ ]:
#@title Step 5b · Optional: train a YOLO detector on the same labels  { display-mode: "form" }
#@markdown SAM 2.1 only follows what it is pointed at, so the automatic pipeline is only as good as the detector that
#@markdown points. A detector trained on your own labels usually finds far more clusters than a text-prompted one.
#@markdown This writes the labelled clips out in YOLO format (same train / val split) and trains an Ultralytics model.
TRAIN_DETECTOR  = False             #@param {type:"boolean"}
DETECTOR_BASE   = "yolo26n-seg.pt"  #@param ["yolo26n-seg.pt", "yolo26s-seg.pt", "yolo26m-seg.pt", "yolo11n-seg.pt", "yolo11s-seg.pt"]
DETECTOR_EPOCHS = 100               #@param {type:"integer"}
DETECTOR_IMGSZ  = 1280              #@param [640, 960, 1280] {type:"raw"}

MY_DETECTOR = None
if TRAIN_DETECTOR:
    MY_DETECTOR = UltralyticsModel(train_detector(DETECTOR_BASE, DETECTOR_EPOCHS, DETECTOR_IMGSZ, f"yolo_{_safe(RUN_NAME)}"),
                                   conf=0.25)
    print("detector ready:", MY_DETECTOR.weights)

## Step 6 · Did it help?
The fine-tuned model is scored on the same val clips with the same hints and added to the leaderboard next to the pretrained rows.

Reading the result:
- **Fine-tuned clearly above pretrained on J&F and on IoU on prompt frame**: it learned our clusters. More labelled clips or more steps may add more.
- **About the same**: pretrained SAM 2.1 was already good at this, there are too few clips, or the labels are SAM 2.1's own uncorrected masks (see *Keeping the score honest* under Step 3). The box rows usually move less than the click rows, because a box leaves less to guess.
- **Worse**: usually too many steps on too few clips (memorising), or labels that disagree with each other. Try fewer steps, or relabel the clips the comparison video shows going wrong.

A difference of a point or two on a handful of val clips is noise. Look at the comparison video before believing a small gap.

In [ ]:
#@title Step 6 · Did fine-tuning help? Same clips, same hints  { display-mode: "form" }
_ft = f"SAM 2.1 {SAM2_SIZE} · fine-tuned ({RUN_NAME})"
LINEUP_FT = {f"{_ft} · box": SAM2Prompted(ckpt=FINETUNED, prompt="box"),
             f"{_ft} · click": SAM2Prompted(ckpt=FINETUNED, prompt="point")}
if INCLUDE_AUTOMATIC:
    LINEUP_FT[f"{Path(DETECTOR_WEIGHTS).stem} '{DETECTOR_TEXT}' → {_ft}"] = DetectThenSAM2(DETECTOR, ckpt=FINETUNED)
if globals().get("MY_DETECTOR") is not None:        # trained in Step 5b
    LINEUP_FT["my YOLO"] = MY_DETECTOR
    LINEUP_FT[f"my YOLO → SAM 2.1 {SAM2_SIZE} pretrained"] = DetectThenSAM2(MY_DETECTOR)
    LINEUP_FT[f"my YOLO → {_ft}"] = DetectThenSAM2(MY_DETECTOR, ckpt=FINETUNED)
board = benchmark(LINEUP_FT)

_val = get_clips("val", labelled=True)
if _val:
    compare(_val[0], [f"SAM 2.1 {SAM2_SIZE} · pretrained · click", f"{_ft} · click"])

## Step 7 · Use it on new video
The automatic pipeline (detector finds clusters, SAM 2.1 segments and follows them) runs over a stretch of video in overlapping chunks, and object ids are carried across chunks. Output: an overlay video, a per-frame count and a table of distinct clusters.

The count here is *clusters the pipeline tracked*, not yield. It inherits every miss and double-count of the detector, and turning cluster counts into kilograms needs a calibration against hand-harvested vines.

In [ ]:
#@title Step 7 · Use it: run on a stretch of new video, get an overlay and a count  { display-mode: "form" }
#@markdown Leave `VIDEO_FILE` empty to use the first video in the videos folder.
VIDEO_FILE    = ""     #@param {type:"string"}
START_SECOND  = 0      #@param {type:"number"}
SECONDS       = 10     #@param {type:"number"}
RUN_FPS       = 6      #@param {type:"number"}
USE_FINETUNED = True   #@param {type:"boolean"}

_videos = sorted(p for p in VIDEOS.iterdir() if p.suffix.lower() in VIDEO_EXTS)
_video = VIDEOS / VIDEO_FILE if VIDEO_FILE else (_videos[0] if _videos else None)
if _video is None:
    print("No video to run on yet.")
else:
    if "DETECTOR" not in globals():
        DETECTOR = UltralyticsModel(DETECTOR_WEIGHTS, text=DETECTOR_TEXT or None, conf=DETECTOR_CONF)
    _finder = globals().get("MY_DETECTOR") or DETECTOR          # your trained detector if there is one
    print("detector:", Path(_finder.weights).name, "· SAM 2.1:", "fine-tuned" if USE_FINETUNED and "FINETUNED" in globals() else "pretrained")
    _pipeline = DetectThenSAM2(_finder, ckpt=FINETUNED if USE_FINETUNED and "FINETUNED" in globals() else None)
    _out, counts, tracks = run_on_video(_pipeline, _video, START_SECOND, SECONDS, RUN_FPS)
    display(Video(str(_out), embed=True, html_attributes="controls loop"))
    display(tracks.head(20))

---
# Appendices
Nothing below runs by itself. These are the extension points.

## Appendix A · Labels from other tools, and labels out to YOLO
For labels you already have, or are making in Roboflow, CVAT or Label Studio. Each labelled image becomes a one-frame clip.

**Import a COCO segmentation export** (Roboflow "COCO Segmentation", CVAT "COCO 1.0", Label Studio "COCO"). Upload and unzip the export, then:

```python
import_coco("/content/export/train/_annotations.coco.json")
import_coco("/content/export/valid/_annotations.coco.json")
# keep only some classes:  import_coco(..., categories=["grape cluster"])
```

**Import YOLO labels** (`images/` and `labels/` folders; outline lines and box lines both work):

```python
import_yolo_seg("/content/export/train/images", "/content/export/train/labels")
```

What happens on import:
- **Outlines** (polygons or masks) are used as they are, and the image counts as labelled.
- **Boxes only** (YOLO detection labels, COCO boxes without a polygon) get a mask drawn by SAM 2.1 from the box. Those images stay *drafts*: open them in `label()`, fix what is wrong and mark them done, or `mark_done([...])` to accept them unchecked. Never put rectangles in as masks: SAM would learn to draw rectangles.
- **Train / val split** is `split="auto"` by default: the video each frame came from is read off its file name (Roboflow's `…_mp4-0012_jpg.rf.<hash>.jpg` style works), and whole videos are held out for val. Roboflow's own train/valid folders mix frames of one video on both sides, which inflates scores, so they are ignored. Pass `split="train"` or `split="val"` to force one.

Single images train SAM 2.1 as an image model and are scored on the *IoU on prompt frame* column and on the automatic table. The tracking columns (J&F) need real clips.

**Export everything labelled here to YOLO format** (same train / val split), for training a detector outside this notebook:

```python
data_yaml = export_yolo_seg(every=2)      # every 2nd frame; neighbouring frames are near-copies
```

Step 5b does this export and the YOLO training in one go. On the HPCC: `python fruit.py import-coco …`, `import-yolo …`, `export-yolo`.

## Appendix B · Scoring SAM 3.1 (or any model that lives in another notebook)
SAM 3.1 needs Meta's `sam3` package (Python 3.12+, PyTorch 2.7+, a CUDA 12.6+ GPU) and Hugging Face access to its gated weights. The safe way to compare it is to keep it in its own notebook or environment and let it **write its masks into this project's `predictions/` folder**. Then it is scored here like everything else.

**1. In the SAM 3.1 notebook**, after `from sam3.model_builder import build_sam3_predictor; predictor = build_sam3_predictor(version="sam3.1", max_num_objects=64)`, paste and run:

```python
import json, numpy as np
from pathlib import Path
from PIL import Image

ROOT = Path("/content/drive/MyDrive/msu_fruit_sam2")     # the same project folder this notebook uses
TEXT = "grape cluster"
OUT  = ROOT / "predictions" / "sam3_text"

clips = json.loads((ROOT / "dataset" / "clips.json").read_text())
for clip, info in clips.items():
    if info.get("split") != "val" or info.get("status") != "done":
        continue
    frames = sorted((ROOT / "dataset" / "JPEGImages" / clip).glob("*.jpg"))
    w, h = Image.open(frames[0]).size
    sid = predictor.handle_request(dict(type="start_session", resource_path=str(frames[0].parent)))["session_id"]
    predictor.handle_request(dict(type="add_prompt", session_id=sid, frame_index=0, text=TEXT))
    out = np.zeros((len(frames), h, w), np.uint8)
    for r in predictor.handle_stream_request(dict(type="propagate_in_video", session_id=sid)):
        for k, m in zip(r["outputs"]["out_obj_ids"].tolist(), r["outputs"]["out_binary_masks"]):
            assert m.shape == (h, w), f"mask is {m.shape}, frame is {(h, w)}"
            out[r["frame_index"]][np.asarray(m, bool)] = min(int(k) + 1, 255)   # 0 stays background
    predictor.handle_request(dict(type="close_session", session_id=sid))
    (OUT / clip).mkdir(parents=True, exist_ok=True)
    for f, m in zip(frames, out):
        Image.fromarray(m).save(OUT / clip / (f.stem + ".png"))                 # pixel value = object id
```

**2. Back here**, run:

```python
benchmark({"SAM 3.1 · text 'grape cluster'": FromFolder(PRED / "sam3_text", tracks=True)})
```

The same two steps work for any model, in any framework: one PNG per frame, same file names as the frames, pixel value = object number, 0 = background.

If `sam3` is installed in *this* runtime instead, `benchmark({"SAM 3.1": SAM3Text("grape cluster")})` calls it directly (or `type: sam3` in a lineup file). Both follow the request/response format in Meta's `sam3` code as of October 2026 (`build_sam3_predictor`, `start_session`, `add_prompt`, `propagate_in_video`) but were **not run** while this pipeline was tested, because the weights are gated. Expect to fix a detail or two. `max_num_objects=64` matters: Meta's default of 16 tracked objects is too few for a vine row.

## Appendix C · Adding your own model in code
A model is a class with one method. It receives a clip and returns one array of object numbers per frame.

```python
class MyModel(Model):
    tracks = False                     # True if the same cluster keeps the same number across frames

    def predict(self, clip, prompts=None):
        out = np.zeros((len(clip), *clip.hw), np.uint16)
        for t in range(len(clip)):
            image = clip.image(t)                      # RGB, uint8, [height, width, 3]
            masks = my_network(image)                  # your code → a list of boolean [height, width] masks
            for number, mask in enumerate(masks, 1):
                out[t][mask] = number
        return out

benchmark({"my model": MyModel()})
```

For a model that needs a hint per object (like SAM 2), set `needs_prompts = True` and `prompt = "box"` (or `"point"`, `"mask"`); `prompts` then arrives as a list of `{"obj", "frame", "box" | "points" | "mask"}`, and the model is scored on the prompted table.

Already wrapped: `SAM2Prompted(size, ckpt, prompt)`, `UltralyticsModel(weights, text)` (YOLO, YOLOE), `DetectThenSAM2(detector, ckpt)`, `FromFolder(folder)`, `SAM3Text(text)`. `clear_leaderboard()` starts the table over.

The same models can be described as plain data, which is how the HPCC jobs take them (`configs/lineup.yaml`, see the README):

```python
benchmark(load_lineup("/path/to/lineup.yaml"))
benchmark({"YOLOE": build_model({"type": "ultralytics", "weights": "yoloe-26m-seg.pt", "text": "grape cluster"})})
```

`examples/my_model.py` is a ready-to-edit template for your own class (`type: python` in a lineup file).

## Appendix D · When something goes wrong

| What you see | What to do |
|---|---|
| "No GPU" under cell B | Runtime ▸ Change runtime type ▸ T4 GPU, then Run all. |
| Step 4 stops with "No labelled 'val' clips yet" | Expected until clips are labelled **and marked done**. Step 3c lists what is missing. |
| The labelling screen is blank or does not react | Runtime ▸ Restart session, Run all. Still stuck: label from code with `ruler("clip_name", frame=0)` to read box corners and `ed = ClipLabels("clip_name"); ed.add_box(0, [x0, y0, x1, y1]); ed.save(done=True)`, or label in Roboflow / CVAT and import (Appendix A). |
| A mask is wrong on some frames | Select the object, **Re-box selected**, drag the right box on that frame. |
| "CUDA out of memory" while scoring | Pick a smaller `SAM2_SIZE` in cell A, or set `SAM2_IMAGE_SIZE = 768` in the SAM 2.1 tools cell (applies to training and scoring alike, so re-train after changing it). |
| Training is very slow | Check cell B found a GPU. Lower `TRAIN_STEPS`. Keep `TRAIN_ENCODER` on "auto" or "no". |
| Colab disconnected | Run all again. Videos, clips, labels and finished weights are on Drive; an interrupted training run resumes from its last save. |
| Fine-tuned model scores worse | See the reading guide under Step 6. Change `RUN_NAME` before trying new settings so results stay comparable. |
| A video will not open | Convert it: `!ffmpeg -i in.mov -c:v libx264 -crf 18 out.mp4`, and put `out.mp4` in the videos folder. |

## Appendix E · The same pipeline on the MSU HPCC
The `sam2_pipeline/` folder of the team repository holds this notebook, `fruit_sam2.py` (the tool cells above as a module), `fruit.py` (a command line around it) and `hpcc/` (setup and job scripts). The README walks through it; the short version:

1. **Once, on a development node** (`ssh dev-amd20-v100`): `git clone -b Saitej https://github.com/moehocbaidi/Capstone-MSU_Fruit.git ~/Capstone-MSU_Fruit`, then `cd ~/Capstone-MSU_Fruit/sam2_pipeline && bash hpcc/setup_env.sh`. That builds the conda environment, fetches the SAM 2 code and the weights, and registers a Jupyter kernel.
2. **Videos** go to `~/msu_fruit_project/videos` (OnDemand *Files ▸ Upload* for files under about 1 GB; Globus or `rsync` to `rsync.hpcc.msu.edu` for big ones). Then `python fruit.py clips` cuts clips and splits them.
3. **Label** in OnDemand: *Interactive Apps ▸ Jupyter*, Jupyter location *Conda Environment using Miniforge3 module* with the environment path `~/envs/fruit_sam2`, one GPU under *Advanced Options*. Open this notebook, set `DEMO` off and `PROJECT_DIR = "~/msu_fruit_project"` in cell A, run the setup and tools cells, then Step 3.
4. **Train and score as batch jobs**, from the code folder:
   ```bash
   sbatch hpcc/run.sb train --run grapes_v1 --steps 3000
   sbatch hpcc/run.sb evaluate
   sbatch hpcc/run.sb video videos/row3.mp4 --seconds 60 --weights models/sam2.1_base_plus_grapes_v1.pt
   ```
   Results land in the project folder, so this notebook (on OnDemand, or on Colab after copying the folder to Drive) shows them too.

## Appendix F · What was checked before this pipeline was handed over, and what was not

Checked on 12–13 Oct 2026 on a CPU-only Linux machine (Python 3.13, PyTorch 2.14, SAM 2 code at commit `2b90b9f`). The runs used the SAM 2.1 **tiny** model at 512 px so they finish on a CPU:

- **This notebook, top to bottom, in demo mode**: setup from scratch, demo download, clips, split, scoring four models, fine-tuning with Meta's trainer, re-scoring, the comparison video, and the run on video.
- **The HPCC command line (`fruit.py`) on a fresh project**:
  - clips from videos, with a whole-video split;
  - approving labels;
  - imports;
  - fine-tuning on video clips and single images together;
  - `evaluate` with every model type in one lineup file: SAM 2.1, YOLOE, YOLOE → SAM 2.1, a folder of PNGs, and a Python class. Fine-tuned rows were added automatically, and a "model" that copies the labels scored exactly 100;
  - `video`, `export-yolo`, `train-detector` and `status`.
- **Imports**:
  - YOLO box labels became SAM 2.1 masks matching the true WGISD masks at mean IoU 0.80 (36 clusters, tiny at 512 px);
  - a COCO box-only object matched at IoU 0.91;
  - outlines were kept as they are;
  - Roboflow-style file names were grouped by source video, and frames of one video never landed on both sides of the split.
- **Scoring**:
  - J (mask IoU) and F (boundary accuracy) agree exactly with Meta's own SA-V/DAVIS evaluator on 965 object-frames;
  - instance matching, in-view averaging and counting were checked against hand-worked cases;
  - scoring the labels against themselves gives 100.
- **Labelling**: boxes, corrective clicks, re-boxing, delete, undo, saving and approval. These were exercised both from code and by driving the real screen with a mouse in JupyterLab.

**Not checked, so treat as unproven until you have run it once:**

- **Anything on a GPU.** On a T4 or V100, training uses float16, because those GPUs lack bfloat16; it switches to bfloat16 by itself if the loss blows up. Speed and memory are estimates, which is why training retries with fewer frames when memory runs out. `base_plus` and `large` were not run at all. They differ from `tiny` only in backbone settings copied from Meta's own configs.
- **Colab itself**: the Drive mount, the upload button, the form fields, and the labelling screen inside Colab's widget system (tested in JupyterLab).
- **The HPCC itself**: `hpcc/setup_env.sh`, `hpcc/run.sb` and OnDemand Jupyter follow ICER's documentation as of October 2026, but were not run on the cluster.
- **`SAM3Text` and the SAM 3.1 snippet** in Appendix B (gated weights).
- **Whether fine-tuning helps on Riesling footage.** That is what Step 6 is for.
- The default training settings (learning rate, steps) are Meta's recipe scaled down, not tuned on vineyard video.